In [1]:
import warnings

# 모든 경고 메시지 무시
warnings.filterwarnings('ignore')

# 1. 설치 검증

In [2]:
import rustima
import numpy as np

print(rustima.version())                              # "0.1.0"
y = np.random.randn(100).cumsum()
r = rustima.sarimax_fit(y, order=(1, 1, 1), seasonal=(0, 0, 0, 0))
print(f"converged={r['converged']}, AIC={r['aic']:.2f}")

0.1.0
converged=True, AIC=295.49


# 2. 기능(with. 예제)

## 2.1. 첫 예측(5분 워크스루)
- converged=True → 최적화 성공
- 낮은 AIC / BIC → 더 나은 모델 적합 (다른 차수 대비 상대적으로)
- Ljung-Box p > 0.05 → 잔차가 백색잡음처럼 보임 (good)
- ci_lower / ci_upper → 불확실성 밴드; 넓을수록 덜 확신

In [3]:
import numpy as np
from rustima import SARIMAXModel, auto_arima

# ── 1. 트렌드 + 1년 계절성이 있는 월별 매출 데이터 시뮬레이션 ────────────
rng = np.random.default_rng(42)
n = 120  # 10년치 월별
trend = 0.5 * np.arange(n)                           # 선형 상승 트렌드
season = 10 * np.sin(2 * np.pi * np.arange(n) / 12)  # 1년 주기 (s=12)
noise = rng.normal(0, 1.0, n)
y = trend + season + noise

# ── 2. auto_arima가 차수를 알아서 선택 ────────────────────────────────────
auto_result = auto_arima(y, s=12, trace=True)  # trace=True → 시도한 모델 출력
print(auto_result.search_summary())
print("\n")
# >>> Best: SARIMA(0,1,1)(0,1,1)[12]  AIC=345.67  (evaluated 23 models)

# ── 3. 선택된 모델 살펴보기 ────────────────────────────────────────────────
model = auto_result.result              # SARIMAXResult 객체
print(model.summary())                  # statsmodels 스타일 파라미터 테이블
print(f"AIC={model.aic:.2f}  BIC={model.bic:.2f}")
print("\n")

# ── 4. 다음 12개월 95% 신뢰구간으로 예측 ──────────────────────────────────
forecast = model.forecast(steps=12, alpha=0.05)
df = forecast.to_dataframe()            # Polars DataFrame
print(df)
# 형태 (12, 5): step | mean | variance | ci_lower | ci_upper

# ── 5. 잔차 검사 (랜덤 노이즈처럼 보여야 함) ──────────────────────────────
diag = model.diagnostics()
print(f"Ljung-Box p-value: {diag['ljung_box_pvalue']:.3f}  (>0.05 이면 good)")

  ARIMA(0,1,0)(0,1,0)[12] : aic=395.332
  ARIMA(2,1,2)(0,1,0)[12] : aic=328.028
  ARIMA(1,1,0)(0,1,0)[12] : aic=361.325
  ARIMA(0,1,1)(0,1,0)[12] : aic=322.248
  ARIMA(0,1,0)(1,1,0)[12] : aic=376.731
  ARIMA(0,1,0)(0,1,1)[12] : aic=369.145
  ARIMA(1,1,0)(1,1,0)[12] : aic=337.221
  ARIMA(0,1,1)(0,1,1)[12] : aic=294.530
  ARIMA(1,1,1)(0,1,1)[12] : aic=296.545
  ARIMA(0,1,2)(0,1,1)[12] : aic=296.574
  ARIMA(0,1,1)(1,1,1)[12] : aic=294.959
  ARIMA(0,1,1)(0,1,2)[12] : aic=295.058
  ARIMA(1,1,2)(0,1,1)[12] : aic=297.474
auto_arima: Best ARIMA(0,1,1)(0,1,1)[12]
  aic=294.530
  Models evaluated: 13 (13 converged)


                               SARIMAX Results                                
Model: SARIMAX(0,1,1)(0,1,1)[12]                  Log Likelihood:     -144.265
No. Observations: 120                              AIC:                294.530
Trend: n                                           BIC:                302.892
Method: lbfgsb                                     HQIC:             

## 2.2 저수준 API

In [4]:
import numpy as np
import polars as pl
import rustima

y = np.random.randn(200).cumsum()

# 1. 모델 적합
result = rustima.sarimax_fit(y, order=(1, 1, 1), seasonal=(0, 0, 0, 0))
print(f"Converged: {result['converged']}, AIC: {result['aic']:.2f}")

# 2. 10스텝 앞 예측
fc = rustima.sarimax_forecast(
    y, order=(1, 1, 1), seasonal=(0, 0, 0, 0),
    params=np.array(result["params"]), steps=10
)
print(f"Forecast: {fc['mean'][:5]}")

# 3. 잔차 진단 (DataFrame 표 형식)
res = rustima.sarimax_residuals(
    y, order=(1, 1, 1), seasonal=(0, 0, 0, 0),
    params=np.array(result["params"])
)

res_df = pl.DataFrame({
    "index": np.arange(len(res["residuals"])),
    "residuals": res["residuals"],
    "standardized_residuals": res["standardized_residuals"],
})
print(res_df)

Converged: True, AIC: 543.35
Forecast: [20.697804464699566, 20.831824162039418, 20.715475186986296, 20.8164833541782, 20.72879328355395]
shape: (200, 3)
┌───────┬───────────┬────────────────────────┐
│ index ┆ residuals ┆ standardized_residuals │
│ ---   ┆ ---       ┆ ---                    │
│ i64   ┆ f64       ┆ f64                    │
╞═══════╪═══════════╪════════════════════════╡
│ 0     ┆ -0.117155 ┆ -0.000089              │
│ 1     ┆ -0.675264 ┆ -0.000616              │
│ 2     ┆ 0.7477    ┆ 0.622298               │
│ 3     ┆ 0.227732  ┆ 0.217278               │
│ 4     ┆ -0.856577 ┆ -0.860833              │
│ …     ┆ …         ┆ …                      │
│ 195   ┆ 0.436967  ┆ 0.467695               │
│ 196   ┆ 0.461229  ┆ 0.493663               │
│ 197   ┆ 0.548777  ┆ 0.587367               │
│ 198   ┆ -0.817285 ┆ -0.874756              │
│ 199   ┆ -0.358794 ┆ -0.384024              │
└───────┴───────────┴────────────────────────┘


## 2.3. 고수준 API(SARIMAXModel - statsmodels 호환)

In [7]:
import numpy as np
import polars as pl
from rustima import SARIMAXModel

model = SARIMAXModel(y, order=(1, 1, 1), seasonal_order=(0, 0, 0, 0), trend="c")
result = model.fit()

print("ㅡ Summaries ㅡ")
# 파라미터 테이블 요약 (빠름, 추론 통계 없음)
print(result.summary())
print("\n")

# Hessian 기반 추론 포함 요약 (std err, z, p-value, CI)
print(result.summary(inference="hessian"))
print("\n")

# Hessian vs statsmodels 추론을 나란히 비교
print(result.summary(inference="both"))
print("\n")

# Polars DataFrame으로 파라미터 테이블
pt = result.params_table(inference="hessian")
print("── Parameter DataFrame ──")
print(pt)  # shape: (k, 7) — name, coef, std_err, z, p_value, ci_lower, ci_upper
print(f"AIC: {result.aic:.2f}, BIC: {result.bic:.2f}, HQIC: {result.hqic:.2f}")
print("\n")

# 신뢰구간 포함 예측 + Polars DataFrame (alpha=0.05 + alpha=0.10 CI 병합)
fcast = result.forecast(steps=10, alpha=0.05)
df = fcast.to_dataframe()  # Polars: step, mean, variance, ci_lower, ci_upper
ci = fcast.conf_int()          # (10, 2) 배열 [lower, upper] — 출력 생략 (df와 중복)
ci_90 = fcast.conf_int(0.10)   # 다른 alpha로 재계산
df = df.with_columns([
    pl.Series("ci_90_lower", ci_90[:, 0]),
    pl.Series("ci_90_upper", ci_90[:, 1]),
])
print("── Forecast DataFrame (alpha=0.05 & alpha=0.1) ──")
print(df)
print("\n")

# In-sample 예측 + 표준화 잔차
pred = result.get_prediction(start=0, end=210)
pred_df = pred.to_dataframe()  # Polars: index, predicted_mean
residuals = result.resid
resid_df = pl.DataFrame({
    "index": np.arange(len(residuals)),
    "resid": residuals,
})
pred_resid_df = pred_df.join(resid_df, on="index", how="left").with_columns(
    pl.when(pl.col("resid").is_null())
      .then(pl.lit("-"))
      .otherwise(pl.col("resid").cast(pl.Utf8))
      .alias("resid")
)
print("── In-sample Prediction + Standardized Residuals ──")
print(pred_resid_df)
print("\n")

# 잔차 진단 (Ljung-Box, Jarque-Bera, 이분산)
diag = result.diagnostics()
diag_flat = {}
for k, v in diag.items():
    if isinstance(v, (list, tuple, np.ndarray)):
        for i, vv in enumerate(np.atleast_1d(v)):
            diag_flat[f"{k}[{i}]"] = float(vv)
    else:
        diag_flat[k] = float(v)

diag_df = pl.DataFrame({
    "value": list(diag_flat.values())
}).transpose(include_header=False)
diag_df.columns = list(diag_flat.keys())
print("── Residual Diagnostics ──")
print(diag_df)

ㅡ Summaries ㅡ
                               SARIMAX Results                                
Model: SARIMAX(1,1,1)(0,0,0)[0]                   Log Likelihood:     -267.498
No. Observations: 200                              AIC:                542.996
Trend: c                                           BIC:                556.190
Method: lbfgsb                                     HQIC:               548.335
Converged: True                                     Scale:            0.860997
Date: 2026-04-21                                                              
------------------------------------------------------------------------------
                       coef
------------------------------------------------------------------------------
       intercept     0.1809
           ar.L1    -0.8610
           ma.L1     0.7974


                               SARIMAX Results                                
Model: SARIMAX(1,1,1)(0,0,0)[0]                   Log Likelihood:     -267.498
No.

## 2.4. auto_arima - 자동 차수 선택

In [10]:
from rustima import auto_arima

print("ㅡ Summary ㅡ")
res = auto_arima(y, max_p=5, max_q=5, s=12, stepwise=True, trace=True)
print(res.summary())           # statsmodels 스타일 전체 요약 + 추론 통계
print("\n")
print("ㅡ Short Summary ㅡ")
print(res.search_summary())    # 짧은 3줄 요약 (차수, IC, 모델 수)
print("\n")
print("ㅡ Forecast(steps=12) ㅡ")
print(res.result.forecast(steps=12).to_dataframe())
print("\n")

# Grid Search (Rayon 병렬 — 차수 조합별 fit job 분산)
res = auto_arima(y, max_p=3, max_q=3, s=7, stepwise=False, criterion="bic")
print("ㅡ Grid Search Summary ㅡ")
print(res.summary())
print("\n")

# 탐색 이력 (Polars DataFrame)
print("ㅡ Grid Search Log ㅡ")
print(res.history_dataframe())

ㅡ Summary ㅡ
  ARIMA(0,1,0)(0,1,0)[12] : aic=626.331
  ARIMA(2,1,2)(0,1,0)[12] : aic=621.169
  ARIMA(1,1,0)(0,1,0)[12] : aic=628.035
  ARIMA(0,1,1)(0,1,0)[12] : aic=628.083
  ARIMA(0,1,0)(1,1,0)[12] : aic=578.452
  ARIMA(0,1,0)(0,1,1)[12] : aic=560.133
  ARIMA(1,1,0)(1,1,0)[12] : aic=577.610
  ARIMA(0,1,1)(0,1,1)[12] : aic=537.675
  ARIMA(1,1,1)(0,1,1)[12] : aic=535.996
  ARIMA(0,1,2)(0,1,1)[12] : aic=538.266
  ARIMA(0,1,1)(1,1,1)[12] : aic=539.277
  ARIMA(0,1,1)(0,1,2)[12] : aic=538.615
  ARIMA(1,1,2)(0,1,1)[12] : aic=537.930
  ARIMA(2,1,1)(0,1,1)[12] : aic=537.934
  ARIMA(1,1,0)(0,1,1)[12] : aic=540.175
  ARIMA(1,1,1)(1,1,1)[12] : aic=537.879
  ARIMA(1,1,1)(0,1,2)[12] : aic=539.214
  ARIMA(1,1,1)(0,1,0)[12] : aic=627.913
  ARIMA(2,1,2)(0,1,1)[12] : aic=537.609
                               SARIMAX Results                                
Model: SARIMAX(1,1,1)(0,1,1)[12]                  Log Likelihood:     -263.998
No. Observations: 200                              AIC:               

## 2.5. Trend(추세) 지원

In [11]:
# 상수항 (intercept)
model = SARIMAXModel(y, order=(1, 1, 1), seasonal_order=(0, 0, 0, 0), trend="c")
# 선형 추세 (drift)
model = SARIMAXModel(y, order=(1, 1, 1), seasonal_order=(0, 0, 0, 0), trend="t")
# 상수 + 선형 (intercept + drift)
model = SARIMAXModel(y, order=(1, 1, 1), seasonal_order=(0, 0, 0, 0), trend="ct")

result = model.fit()
print(result.param_names)  # ['intercept', 'drift', 'ar.L1', 'ma.L1'] (trend='ct')

['intercept', 'drift', 'ar.L1', 'ma.L1']


## 2.6. 외생 회귀변수 사용

In [12]:
import numpy as np

# 외생 회귀변수(2개) 생성
X_train = np.column_stack([np.arange(200), np.random.randn(200)])  # (200, 2)
X_future = np.column_stack([np.arange(200, 210), np.random.randn(10)])  # (10, 2)

model = SARIMAXModel(y, order=(1, 0, 1), seasonal_order=(0, 0, 0, 0), exog=X_train)
result = model.fit()
fcast = result.forecast(steps=10, exog=X_future)

## 2.7. 배치 병렬 처리

In [13]:
# 100개 시계열을 job 단위로 병렬 적합 (Rayon 멀티스레드)
series_list = [np.random.randn(200) for _ in range(100)]

results = rustima.sarimax_batch_fit(
    series_list, order=(1, 0, 0), seasonal=(0, 0, 0, 0)
)

for i, r in enumerate(results):
    print(f"Series {i}: converged={r['converged']}, AIC={r['aic']:.2f}")

# 시계열별 파라미터로 배치 예측
params_list = [np.array(r["params"]) for r in results]
forecasts = rustima.sarimax_batch_forecast(
    series_list, order=(1, 0, 0), seasonal=(0, 0, 0, 0),
    params_list=params_list, steps=10, alpha=0.05,
)

Series 0: converged=True, AIC=597.88
Series 1: converged=True, AIC=562.84
Series 2: converged=True, AIC=590.44
Series 3: converged=True, AIC=541.87
Series 4: converged=True, AIC=584.02
Series 5: converged=True, AIC=600.57
Series 6: converged=True, AIC=574.77
Series 7: converged=True, AIC=568.01
Series 8: converged=True, AIC=566.87
Series 9: converged=True, AIC=572.59
Series 10: converged=True, AIC=583.63
Series 11: converged=True, AIC=621.42
Series 12: converged=True, AIC=570.22
Series 13: converged=True, AIC=547.39
Series 14: converged=True, AIC=554.94
Series 15: converged=True, AIC=555.65
Series 16: converged=True, AIC=580.07
Series 17: converged=True, AIC=564.01
Series 18: converged=True, AIC=577.36
Series 19: converged=True, AIC=571.78
Series 20: converged=True, AIC=583.30
Series 21: converged=True, AIC=569.08
Series 22: converged=True, AIC=548.63
Series 23: converged=True, AIC=533.17
Series 24: converged=True, AIC=583.06
Series 25: converged=True, AIC=559.14
Series 26: converged=T

## 2.8. Grid Search 병렬 처리

In [14]:
# 여러 ARIMA 차수를 Rayon으로 한꺼번에 적합
results = rustima.sarimax_grid_search(
    y,
    order_list=[(0,1,0), (1,1,0), (1,1,1), (2,1,1)],
    seasonal_list=[(0,0,0,0)] * 4,
    trend="c",
)
for r in results:
    if "error" not in r:
        print(f"{r['order']}: AIC={r['aic']:.3f}")

(0, 1, 0): AIC=541.589
(1, 1, 0): AIC=542.932
(1, 1, 1): AIC=542.996
(2, 1, 1): AIC=544.853


# 3. Python API
- Python에서 rustima를 사용할 때 호출하는 함수 및 클래스 집합 

## 3.1. 저수준 함수

### 3.1.1. rustima.sarimax_loglike
- 주어진 파라미터에서 로그우도 계산

In [18]:
ll = rustima.sarimax_loglike(
    y,
    order=(1, 1, 1),           # (p, d, q)
    seasonal=(1, 1, 1, 12),    # (P, D, Q, s)
    params=np.array([8.0, 0.5, 0.3, 0.2, -0.4]),    # [ar, ma, sar, sma]
    concentrate_scale=True,    # 우도에서 sigma2를 집중화
    trend="c",                 # 추세: "n", "c", "t", "ct"
)

ll

-698.9136172127642

### 3.1.2. rustima.sarimax_fit
- MLE로 모델 적합

In [22]:
result = rustima.sarimax_fit(
    y,
    order=(1, 0, 1),
    seasonal=(0, 0, 0, 0),
    enforce_stationarity=True,   # AR 정상성 제약
    enforce_invertibility=True,  # MA 가역성 제약
    method="lbfgsb",             # "lbfgsb" | "lbfgsb-multi" | "lbfgs" | "nelder-mead"
    maxiter=500,
    trend="c",                   # 추세
)

result

{'params': [0.15624264155326809, 0.9941585002004081, -0.05250547183298274],
 'loglike': -269.1681355119009,
 'scale': 0.8639885943142959,
 'aic': 546.3362710238018,
 'bic': 559.529540489994,
 'n_obs': 200,
 'n_params': 4,
 'n_iter': 54,
 'converged': True,
 'method': 'lbfgsb',
 'warnings': []}

### 3.1.3. rustima.sarimax_forecast
- 신뢰구간과 함께 h-step 앞 예측 수행

In [21]:
fc = rustima.sarimax_forecast(
    y,
    order=(1, 0, 0),
    seasonal=(0, 0, 0, 0),
    params=np.array([8.0, 0.42, 0.33, 0.65]), # [intercept, exog, future_exog, ar]
    steps=10,          # 예측 구간
    alpha=0.05,        # 95% 신뢰구간
    exog=X_train,      # 모델이 exog를 쓰는 경우 과거 exog
    future_exog=X_future,  # 예측 기간 미래 exog
    trend="c",
)

print(fc["mean"])       # 점예측 (list[float])
print(fc["ci_lower"])   # 하한
print(fc["ci_upper"])   # 상한
print(fc["variance"])   # 예측 분산

[51.79326274672786, 70.86558127650473, 84.17188695582811, 93.22782013890198, 99.33570621331913, 102.77353737252352, 105.4307230645238, 107.65926997376928, 108.55496185303396, 110.09221121953678]
[10.889045520842451, 22.079692242770875, 32.41542271042778, 40.266333338225266, 45.873258645789704, 49.100838948885944, 51.669440743470474, 53.860604787613305, 54.74051020997587, 56.271091189881105]
[92.69747997261328, 119.65147031023858, 135.92835120122845, 146.18930693957867, 152.79815378084857, 156.4462357961611, 159.19200538557715, 161.45793515992526, 162.36941349609205, 163.91333124919245]
[435.55197706434535, 619.5726873740313, 697.3214374798737, 730.170284399592, 744.048922223173, 749.912646703636, 752.3900702966315, 753.4367817646722, 753.8790173599193, 754.0658618989112]


### 3.1.4. rustima.sarimax_residuals
- 잔차와 표준화 잔차 계산

In [23]:
res = rustima.sarimax_residuals(
    y,
    order=(1, 0, 1),
    seasonal=(0, 0, 0, 0),
    params=np.array([8.0, 0.5, 0.3]),   # [intercept, ar, ma]
    trend="c",
)

print(res["residuals"])                # 혁신항 v_t
print(res["standardized_residuals"])   # v_t / sqrt(F_t * sigma2)

[-0.11715474777972851, -8.682987160204831, -7.080999303305964, -6.3034142764926475, -6.365272054048714, -6.135117439203218, -5.268695135127875, -6.447470385178158, -7.472943259770094, -7.299010152150907, -6.558391636070291, -5.663474719344281, -5.053302295348735, -5.611600844203753, -6.400870016759525, -6.170287344815487, -7.145621573041378, -5.846678978076188, -5.183784987459902, -4.694517502692085, -4.326474030467539, -6.4007822944688195, -5.728512603825986, -6.195643687479511, -4.012077141927666, -6.4084533164654065, -6.976387689417768, -5.319681564080537, -6.9248998549885865, -6.639000930113665, -7.573556808678678, -6.1848791473146445, -6.069827166938316, -7.32724459404176, -6.378576111277097, -6.654504998345342, -7.426131151807349, -5.90743439695224, -6.209769990349204, -6.853037329278389, -4.637819834217398, -6.856504978841042, -4.882394523353315, -5.023261343399602, -5.841121186149911, -6.562314726785063, -4.6908182294232805, -5.8835913291684685, -4.77242620214818, -6.9059328060

### 3.1.5. rustima.sarimax_batch_fit
- Rayon thread 풀을 이용해 N개의 시계열을 병렬 적합. 각 worker는 시계열 하나에 대한 전체 적합 작업 수행

In [25]:
results = rustima.sarimax_batch_fit(
    series_list,
    order=(1, 0, 0),
    seasonal=(0, 0, 0, 0),
    enforce_stationarity=True,
    method="lbfgsb",
    maxiter=500,
    trend="c",
)
# 반환: list[dict] — sarimax_fit과 동일 키
# 실패한 시계열: {"error": "...", "converged": false}

results

[{'params': [-0.046564128391393046, 0.03955737016981842],
  'loglike': -296.74936641219216,
  'scale': 1.1383918376091327,
  'aic': 599.4987328243843,
  'bic': 609.3936849240284,
  'n_obs': 200,
  'n_params': 3,
  'n_iter': 5,
  'converged': True,
  'method': 'lbfgsb',
  'warnings': []},
 {'params': [-0.006825370643713457, 0.01397968470559936],
  'loglike': -279.4130992133762,
  'scale': 0.9571969835539126,
  'aic': 564.8261984267524,
  'bic': 574.7211505263965,
  'n_obs': 200,
  'n_params': 3,
  'n_iter': 5,
  'converged': True,
  'method': 'lbfgsb',
  'warnings': []},
 {'params': [0.0108865619459793, 0.15940068662360962],
  'loglike': -293.21012730523944,
  'scale': 1.0988060779235984,
  'aic': 592.4202546104789,
  'bic': 602.315206710123,
  'n_obs': 200,
  'n_params': 3,
  'n_iter': 5,
  'converged': True,
  'method': 'lbfgsb',
  'warnings': []},
 {'params': [-0.032538433879394306, -0.0011939442408053758],
  'loglike': -268.81266806758725,
  'scale': 0.8609228482317056,
  'aic': 543

### 3.1.6. rustima.sarimax_batch_forecast
- N개 시계열(각기 다른 파라미터) 병렬 에측
- 각 워커는 시계열 하나에 대한 전체 예측 작업 수행

In [26]:
params_list = [np.array(r["params"]) for r in results]

forecasts = rustima.sarimax_batch_forecast(
    series_list,
    order=(1, 0, 0),
    seasonal=(0, 0, 0, 0),
    params_list=params_list,
    steps=10,
    alpha=0.05,
)
# 반환: mean, variance, ci_lower, ci_upper를 포함한 list[dict]

params_list

[array([-0.04656413,  0.03955737]),
 array([-0.00682537,  0.01397968]),
 array([0.01088656, 0.15940069]),
 array([-0.03253843, -0.00119394]),
 array([ 0.00038836, -0.01945915]),
 array([-0.00458029, -0.04283344]),
 array([ 0.08495406, -0.05365668]),
 array([ 0.1062901, -0.0728362]),
 array([-0.00782035, -0.06383649]),
 array([0.03689452, 0.0412186 ]),
 array([-0.04969349, -0.0592132 ]),
 array([-0.14176786, -0.08525005]),
 array([ 0.0509174 , -0.05769727]),
 array([0.15943192, 0.00730548]),
 array([0.0087015 , 0.03566506]),
 array([-0.01815473,  0.0491498 ]),
 array([-0.05083933, -0.15254071]),
 array([-0.05730866,  0.00344505]),
 array([0.07020907, 0.05186902]),
 array([ 0.084015  , -0.04709703]),
 array([-0.11804668, -0.04937679]),
 array([0.02127029, 0.00225742]),
 array([ 0.0192645 , -0.01990367]),
 array([-0.18276503, -0.06603538]),
 array([0.12617639, 0.03891807]),
 array([-0.0626074 , -0.10051214]),
 array([-0.0316304 , -0.01360072]),
 array([-0.08936421, -0.03684538]),
 array([

### 3.1.7. rustima.sarimax_grid_search
- 단일 시계열에 여러 ARIMA 차수 조합을 Rayon 병렬로 적합
- 각 워커는 차수 조합 하나에 대한 전체 적합 작업 수행


In [28]:
results = rustima.sarimax_grid_search(
    y,
    order_list=[(1,0,0), (1,0,1), (2,0,0)],
    seasonal_list=[(0,0,0,0)] * 3,
    enforce_stationarity=True,
    enforce_invertibility=True,
    trend="c",
    method="lbfgsb",
    maxiter=500,
)
# 반환: list[dict] — sarimax_fit과 동일 키 + "order", "seasonal_order"

results

[{'params': [0.16635468437218495, 0.9929725774845125],
  'loglike': -269.4392014475385,
  'scale': 0.8663337501045655,
  'aic': 544.878402895077,
  'bic': 554.7733549947211,
  'n_obs': 200,
  'n_params': 3,
  'n_iter': 21,
  'converged': True,
  'method': 'lbfgsb',
  'warnings': [],
  'order': (1, 0, 0),
  'seasonal_order': (0, 0, 0, 0)},
 {'params': [0.15624264155326809, 0.9941585002004081, -0.05250547183298274],
  'loglike': -269.1681355119009,
  'scale': 0.8639885943142959,
  'aic': 546.3362710238018,
  'bic': 559.529540489994,
  'n_obs': 200,
  'n_params': 4,
  'n_iter': 54,
  'converged': True,
  'method': 'lbfgsb',
  'warnings': [],
  'order': (1, 0, 1),
  'seasonal_order': (0, 0, 0, 0)},
 {'params': [0.16419251877618682, 0.9413413139958462, 0.05251946397533497],
  'loglike': -269.1658806130006,
  'scale': 0.8639691124646325,
  'aic': 546.3317612260012,
  'bic': 559.5250306921934,
  'n_obs': 200,
  'n_params': 4,
  'n_iter': 26,
  'converged': True,
  'method': 'lbfgsb',
  'warni

### 3.1.8. rustima.sarimax_inferaence
- 적합된 파라미터에서 Hessioan 또는 OPG 기반 추론 통계 계산

In [51]:
inf = rustima.sarimax_inference(
    y, order=(1,0,1), seasonal=(0,0,0,0),
    params=np.array([0.15, 0.99, 0.05]),    # [intercept, ar, ma]
    method="hessian",   # "hessian" | "opg"
    alpha=0.05,
    trend="c",
)
print(inf["std_err"])   # 표준오차
print(inf["z_stat"])    # z 통계량
print(inf["p_value"])   # 양측 p-value
print(inf["ci_lower"])  # 신뢰구간 하한
print(inf["ci_upper"])  # 신뢰구간 상한

[0.0919229002553244, 881.5950728769077, 0.06887753749981854]
[1.6318022993547967, 0.0011229645337845805, 0.7259260684244371]
[0.10272114080053929, 0.9991040041244799, 0.46788409463670466]
[-0.030165573854903566, -1726.9045917867031, -0.0849974928434513]
[0.3301655738549035, 1728.8845917867031, 0.1849974928434513]


### 3.1.9. rustima.sarimax_diagnostics
- 잔차 진단 검정 수행

In [49]:
diag = rustima.sarimax_diagnostics(
    y, order=(1,0,1), seasonal=(0,0,0,0),
    params=np.array([0.4, 0.5, 0.3]),   # [intercept, ar, ma]
    trend="c",
)
print(diag["ljung_box_stat"])     # Ljung-Box Q 통계량
print(diag["ljung_box_pvalue"])   # Ljung-Box p-value
print(diag["jarque_bera_stat"])   # Jarque-Bera 정규성
print(diag["het_stat"])           # 이분산 검정

1289.9550669764433
0.0
5.1771887567786825
1.3219161896910334


## 3.2. 고수준 클래스
- Rust 엔진을 사용하는 statsmodels 호환 Python 래퍼

### 3.2.1. SARIMAXModel

In [58]:
from rustima import SARIMAXModel

model = SARIMAXModel(
    endog=y,                        # 시계열 데이터
    order=(1, 1, 1),                # ARIMA(p, d, q)
    seasonal_order=(1, 0, 0, 12),   # (P, D, Q, s)
    trend="c",                      # 추세: 'n', 'c', 't', 'ct'
    enforce_stationarity=True,
    enforce_invertibility=True,
)

model_exog = SARIMAXModel(
    endog=y,                        # 시계열 데이터
    order=(1, 1, 1),                # ARIMA(p, d, q)
    seasonal_order=(1, 0, 0, 12),   # (P, D, Q, s)
    exog=X_train,                   # 외생 회귀변수
    trend="c",                      # 추세: 'n', 'c', 't', 'ct'
    enforce_stationarity=True,
    enforce_invertibility=True,
)

### 3.2.2. SARIMAXResult
- model.fit()의 반환 객체

In [64]:
result = model.fit(method="lbfgsb", maxiter=500)
result_exog = model_exog.fit(method="lbfgsb", maxiter=500)

# 속성
print(result.params)          # np.ndarray — 추정 파라미터
print(result.param_names)     # list[str] — 파라미터 이름 (예: ['intercept', 'ar.L1', 'ma.L1'])
print(result.llf)             # float — 로그우도
print(result.aic)             # float — AIC
print(result.bic)             # float — BIC
print(result.hqic)            # float — HQIC (Hannan-Quinn)
print(result.scale)           # float — sigma2
print(result.nobs)            # int — 관측치 개수
print(result.converged)       # bool — 수렴 상태
print(result.method)          # str — 최적화 방법
print(result.resid)           # np.ndarray — 표준화 잔차(지연 계산)

# 메서드
result.forecast(steps=10, alpha=0.05)     # → ForecastResult
result_exog.forecast(steps=10, exog=X_future)  # 미래 exog 포함(이 때는 exog)
result.get_forecast(steps=10, alpha=0.05) # alias (statsmodels 호환)
result.get_prediction(start=0, end=210)   # → PredictionResult (in-sample + out-of-sample)
print(result.summary())                          # → str (기본 파라미터 테이블)
print(result.summary(inference="hessian"))       # → str (std err / z / p / CI 포함)
print(result.summary(inference="statsmodels"))   # → str (statsmodels 추론값 차용)
print(result.summary(inference="both"))          # → str (양쪽 비교)
print(result.params_table(inference="hessian"))  # → Polars DataFrame
print(result.diagnostics())                      # → dict (Ljung-Box, Jarque-Bera, 이분산)

[ 0.17180101 -0.85143993  0.78686269  0.0551561 ]
['intercept', 'ar.L1', 'ma.L1', 'ar.S.L12']
-267.22193469024563
544.4438693804913
560.9354562132314
551.1177623019058
0.8584224824345392
200
True
lbfgsb
[-8.95941005e-05 -7.83943127e-04  6.34129661e-04  4.32551133e-04
 -9.44449607e-04 -3.40819207e-04  9.30333965e-04  2.38760729e-04
 -2.31127313e-03 -2.66310128e-03 -6.90152248e-04  1.50023508e-03
  2.62242976e-03  5.80693538e-02 -5.54811385e-01 -9.00227746e-01
 -9.97435375e-01  4.34567651e-02  1.49633562e+00  1.23722777e+00
  1.45726186e+00 -1.58207123e+00 -8.70808500e-01 -8.12503463e-01
  1.65158954e+00 -9.07020111e-01 -2.10258057e+00  6.23919151e-01
 -8.73977957e-01 -8.42980319e-01 -1.40696341e+00  3.01513334e-01
  7.97009098e-01 -9.44895927e-01  8.03824388e-02  6.11799051e-02
 -1.10330840e+00  8.53109426e-01  7.44596977e-01 -6.46821923e-01
  1.88742399e+00 -6.39490837e-01  8.77748141e-01  1.07380231e+00
 -6.58205765e-01 -1.27306037e+00  9.41372295e-01 -5.20073316e-02
  5.76298581e-01 

### 3.2.3. ForecastResult

In [70]:
fcast = result.forecast(steps=10)

print(fcast.predicted_mean)   # np.ndarray — 점예측
print("\n")
print(fcast.variance)         # np.ndarray — 예측 분산
print("\n")
print(fcast.ci_lower)         # np.ndarray — 신뢰구간 하한
print("\n")
print(fcast.ci_upper)         # np.ndarray — 신뢰구간 상한
print("\n")

print(fcast.conf_int())       # np.ndarray (steps, 2) — 원래 alpha 기준 [lower, upper]
print("\n")
print(fcast.conf_int(0.10))   # 다른 유의수준으로 재계산
print("\n")
print(fcast.to_dataframe())   # Polars DataFrame (step, mean, variance, ci_lower, ci_upper)

[20.81704259 21.0571301  21.0583049  21.32573639 21.36956908 21.61486583
 21.57444522 21.74898643 21.81649168 21.98143522]


[0.8549308  1.60300872 2.44161448 3.20281429 4.02968361 4.80046747
 5.61888017 6.39664947 7.20895873 7.99181183]


[19.00481151 18.57562008 17.99573057 17.81810482 17.43512328 17.32059079
 16.92851186 16.79192424 16.55408556 16.44065765]


[22.62927367 23.53864012 24.12087922 24.83336796 25.30401487 25.90914087
 26.22037859 26.70604863 27.0788978  27.52221279]


[[19.00481151 22.62927367]
 [18.57562008 23.53864012]
 [17.99573057 24.12087922]
 [17.81810482 24.83336796]
 [17.43512328 25.30401487]
 [17.32059079 25.90914087]
 [16.92851186 26.22037859]
 [16.79192424 26.70604863]
 [16.55408556 27.0788978 ]
 [16.44065765 27.52221279]]


[[19.29617033 22.33791485]
 [18.97458124 23.13967896]
 [18.48811151 23.62849829]
 [18.38203918 24.2694336 ]
 [18.06767808 24.67146007]
 [18.01099661 25.21873505]
 [17.67545504 25.47343541]
 [17.5888887  25.90908416]
 [17.40014125 26.2328

### 3.2.4. PredictionResult

In [71]:
pred = result.get_prediction(start=0, end=210)

print(pred.predicted_mean)    # np.ndarray — 예측값 (in-sample + out-of-sample)
print("\n")
print(pred.to_dataframe())    # Polars DataFrame (index, predicted_mean)

[ 0.          0.10452138 -0.03811659 -0.37815319  0.64181198  0.15597286
 -0.04890954  0.15681577  0.91866848  0.31228142 -1.18500723 -1.92971749
 -1.4497441  -0.16757779  1.24855532  1.13303714  0.08235263 -0.47834069
 -0.56588052  1.01534021  1.99463191  3.445875    2.14695808  1.50811241
  0.9784619   2.48061439  1.86662919 -0.02872121  0.561565   -0.06028752
 -0.68417105 -1.72832018 -1.38700273 -0.59953371 -1.38768328 -1.28132611
 -1.014712   -1.94151196 -1.24682608 -0.39707663 -0.93646292  0.74778659
  0.32731495  1.08860742  2.30122957  1.70877012  0.73590479  1.5604109
  1.68579832  2.21895748  1.11088667  1.47352727  0.78481688  0.23152657
  0.68989091  2.1245479   3.00983985  3.91890619  3.24521071  4.3720293
  6.57133139  7.01671259  7.3379243   6.77671626  7.47759556  8.090949
  8.28719478  6.34391251  6.47722216  5.96130037  5.36961567  6.61673272
  8.13263436  8.95304878  9.43313057  8.52655011  9.12222691  9.42419914
  9.70777657 10.47325301 11.21069819 10.72667638 10.629

### 3.2.5. AutoARIMAResult

In [73]:
from rustima import auto_arima

res = auto_arima(y, max_p=5, max_q=5, s=12)

res.result             # SARIMAXResult — 최적 모델 적합 결과
print(res.order)              # tuple (p, d, q) — 최적 차수
print("\n")
print(res.seasonal_order)     # tuple (P, D, Q, s)
print("\n")
print(res.best_ic)            # float — 최적 정보기준 값
print("\n")
print(res.criterion)          # str — 사용된 기준 ("aic", "bic", "hqic")
print("\n")
print(res.history)            # list[dict] — 탐색 이력
print("\n")
print(res.summary())        # str — statsmodels 스타일 전체 요약 + 추론 통계
print("\n")
print(res.search_summary())   # str — 짧은 3줄 요약 (차수, IC, 모델 수)
print("\n")
print(res.history_dataframe())  # Polars DataFrame — 탐색 이력 테이블

(1, 1, 1)


(0, 1, 1, 12)


535.9963436337109


aic


[{'order': (0, 1, 0), 'seasonal_order': (0, 1, 0, 12), 'aic': 626.3307095151272, 'converged': True}, {'order': (2, 1, 2), 'seasonal_order': (0, 1, 0, 12), 'aic': 621.1688399827144, 'converged': True}, {'order': (1, 1, 0), 'seasonal_order': (0, 1, 0, 12), 'aic': 628.0351560987984, 'converged': True}, {'order': (0, 1, 1), 'seasonal_order': (0, 1, 0, 12), 'aic': 628.0828880277735, 'converged': True}, {'order': (0, 1, 0), 'seasonal_order': (1, 1, 0, 12), 'aic': 578.451811118033, 'converged': True}, {'order': (0, 1, 0), 'seasonal_order': (0, 1, 1, 12), 'aic': 560.1333130425375, 'converged': True}, {'order': (1, 1, 0), 'seasonal_order': (1, 1, 0, 12), 'aic': 577.6104566432962, 'converged': True}, {'order': (0, 1, 1), 'seasonal_order': (0, 1, 1, 12), 'aic': 537.6748802684077, 'converged': True}, {'order': (1, 1, 1), 'seasonal_order': (0, 1, 1, 12), 'aic': 535.9963436337109, 'converged': True}, {'order': (0, 1, 2), 'seasonal_order': (0, 1, 

# sarimax_rs vs statsmodels 비교 분석

두 패키지로 fitting한 뒤,
- 로그우도(log-likelihood) 유사도
- 수렴 속도(fitting time)
- 파라미터 일치도

를 비교합니다.

**데이터**
- 시간단위: `smp_land_hourly.csv`

In [2]:
import numpy as np
import polars as pl
import time
import warnings
from rustima import SARIMAXModel, auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX as SM_SARIMAX
import pmdarima as pm
import matplotlib.pyplot as plt

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 경고 무시
warnings.filterwarnings('ignore')

## 1. 데이터 로드 및 전처리

In [3]:
df_hourly = pl.read_csv("test_data/smp_land_hourly.csv")
print(f"\n시간단위 데이터: {df_hourly.shape}")
print(df_hourly.head())


시간단위 데이터: (218568, 3)
shape: (5, 3)
┌────────────┬──────────┬───────┐
│ date       ┆ time     ┆ price │
│ ---        ┆ ---      ┆ ---   │
│ str        ┆ str      ┆ f64   │
╞════════════╪══════════╪═══════╡
│ 2001-05-01 ┆ 01:00:00 ┆ 51.69 │
│ 2001-05-01 ┆ 02:00:00 ┆ 51.59 │
│ 2001-05-01 ┆ 03:00:00 ┆ 41.16 │
│ 2001-05-01 ┆ 04:00:00 ┆ 14.4  │
│ 2001-05-01 ┆ 05:00:00 ┆ 14.24 │
└────────────┴──────────┴───────┘


In [4]:
df_hourly = (                                                                                                                        
    df_hourly
    .with_columns([
        (pl.col("time") == "24:00:00").alias("_midnight"),
        pl.when(pl.col("time") == "24:00:00")
            .then(pl.lit("00:00:00"))
            .otherwise(pl.col("time"))
            .alias("_time_fix"),
    ])
    .with_columns(
        pl.concat_str(["date", "_time_fix"], separator=" ")
            .str.to_datetime()
            .alias("datetime")
    )
    .with_columns(
        pl.when(pl.col("_midnight"))
            .then(pl.col("datetime") + pl.duration(days=1))
            .otherwise(pl.col("datetime"))
            .alias("datetime")
    )
    .drop(["_midnight", "_time_fix", "date", "time"])
    .filter((pl.col("datetime") >= pl.datetime(2021, 4, 1, 1, 0, 0)) & (pl.col("datetime") <= pl.datetime(2026, 3, 31, 0, 0, 0)))
  )
print(df_hourly)

shape: (43_800, 2)
┌────────┬─────────────────────┐
│ price  ┆ datetime            │
│ ---    ┆ ---                 │
│ f64    ┆ datetime[μs]        │
╞════════╪═════════════════════╡
│ 74.05  ┆ 2021-04-01 01:00:00 │
│ 73.3   ┆ 2021-04-01 02:00:00 │
│ 72.57  ┆ 2021-04-01 03:00:00 │
│ 72.57  ┆ 2021-04-01 04:00:00 │
│ 73.79  ┆ 2021-04-01 05:00:00 │
│ …      ┆ …                   │
│ 119.67 ┆ 2026-03-30 20:00:00 │
│ 114.56 ┆ 2026-03-30 21:00:00 │
│ 113.53 ┆ 2026-03-30 22:00:00 │
│ 111.97 ┆ 2026-03-30 23:00:00 │
│ 109.24 ┆ 2026-03-31 00:00:00 │
└────────┴─────────────────────┘


In [5]:
print(len(df_hourly))

43800


In [6]:
y = df_hourly["price"].to_numpy().astype(np.float64)

# 2. rustima vs pmdarima 비교 

- `seasonal × stepwise × criterion × trend` = **2 × 2 × 3 × 4 = 48 조합**, rustima vs pmdarima 2개로 **총 96런** 자동 실행

- 각 런마다 50 ms 간격으로 RSS 를 샘플링한 **메모리 타임라인(JSON)** 과 `trace=True` **stdout 로그(.log)** 를 저장
- 전체 결과는 `bench_results/sweep_<timestamp>/sweep_results.csv` 로 누적 저장되므로, 중간에 중단돼도 일부 결과가 유지
- 전체 조합에 대한 결과를 한눈에 볼 수 있는 table 3개 생성
- **psutil 샘플러**를 통해 **메모리 타임라인(JSON)** 시각화(대시보드, 히트맵, 타임라인, HTML 리포트)
- 파라미터 조합에 따라 커널 shutdown이 일어날 수 있으므로 각 런을 `try/except` 로 감싸고, 실패 런도 CSV 에 `status=fail` 로 기록

## 2.1. 전체 조합 실행(sweep run)
- 전체 조합 스위프 — subprocess 격리 + Resume 지원 (커널 크래시 방어)
- 방어선
    - 각 런을 별도 subprocess 로 실행 → OOM/segfault 가 나도 메인 커널은 살아남음
    - 완료분은 CSV 에 즉시 기록 → 재실행 시 자동 스킵 (resume)
    - y 를 y.pkl 로 디스크에 보존 → 커널이 죽어도 이 셀만 다시 실행하면 됨
    - per-run timeout → 먹통 런 강제 종료 후 다음으로 진행
    - 워커가 10 샘플마다 mem_json 을 flush → 크래시 직전까지 타임라인 보존

In [7]:
# ======================================================================
# sweep-clean — sweep-run 실행 전 기존 산출물 정리
#
# MODE 를 바꿔서 원하는 수준의 정리를 실행하세요.
#   "none"  → 아무것도 안 함 (현재 상태만 출력)
#   "pkl"   → sweep_*/y.pkl 만 삭제 (sweep-run 이 y 를 새로 pickle)
#   "dir"   → sweep_* 디렉터리 전체 삭제 (완전 초기화; RESUME 무효)
#   "latest"→ 가장 최근 sweep_* 디렉터리만 통째로 삭제
#
# 실수 방지를 위해 DRY_RUN=True 이면 삭제 대상만 출력하고 실행 안 함.
# ======================================================================
import shutil
from pathlib import Path

MODE = "pkl"       # "none" | "pkl" | "dir" | "latest"
DRY_RUN = False    # True 이면 삭제 대상만 표시

base = Path("bench_results")
if not base.exists():
    print("bench_results 디렉터리 없음 — 정리할 것 없음")
else:
    sweep_dirs = sorted(base.glob("sweep_*"))
    print(f"현재 sweep 디렉터리: {len(sweep_dirs)}개")
    for d in sweep_dirs:
        csv_p = d / "sweep_results.csv"
        pkl_p = d / "y.pkl"
        n_rows = 0
        if csv_p.exists():
            try:
                n_rows = sum(1 for _ in open(csv_p, encoding="utf-8")) - 1
            except Exception:
                pass
        print(f"  {d.name}  rows={n_rows:3d}  y.pkl={'O' if pkl_p.exists() else '-'}")

    def _rm_file(p):
        print(f"  rm  {p}")
        if not DRY_RUN:
            try: p.unlink()
            except OSError as e: print(f"     (failed: {e})")

    def _rm_tree(p):
        print(f"  rmtree  {p}")
        if not DRY_RUN:
            try: shutil.rmtree(p)
            except OSError as e: print(f"     (failed: {e})")

    if MODE == "none":
        print("\nMODE=none — 건너뜀")
    elif MODE == "pkl":
        if "y" not in globals():
            print("⚠️  [MODE=pkl] 커널에 y 가 없습니다. y.pkl 을 지우면 "
                  "RESUME 시 sweep-run 이 RuntimeError 로 실패합니다.")
            print("    먼저 y 정의 셀을 실행한 뒤 다시 시도하거나, "
                  "전체 초기화가 목적이면 MODE=\"latest\" 또는 \"dir\" 을 쓰세요.")
        else:
            targets = [p for d in sweep_dirs for p in d.glob("*.pkl")]
            print(f"[MODE=pkl] {len(targets)}개 pkl 삭제 (DRY_RUN={DRY_RUN})")
            for p in targets:
                _rm_file(p)
    elif MODE == "dir":
        print(f"\n[MODE=dir] {len(sweep_dirs)}개 sweep 디렉터리 전체 삭제 (DRY_RUN={DRY_RUN})")
        for d in sweep_dirs:
            _rm_tree(d)
    elif MODE == "latest":
        if sweep_dirs:
            print(f"\n[MODE=latest] 가장 최근 1개 삭제 (DRY_RUN={DRY_RUN})")
            _rm_tree(sweep_dirs[-1])
        else:
            print("\n[MODE=latest] 대상 없음")
    else:
        raise ValueError(f"알 수 없는 MODE: {MODE!r}")

    if not DRY_RUN and MODE != "none":
        print(f"\n✅ 정리 완료. 남은 sweep 디렉터리: "
              f"{len(list(base.glob('sweep_*')))}개")

현재 sweep 디렉터리: 1개
  sweep_20260424_202726  rows=  0  y.pkl=O
[MODE=pkl] 1개 pkl 삭제 (DRY_RUN=False)
  rm  bench_results\sweep_20260424_202726\y.pkl

✅ 정리 완료. 남은 sweep 디렉터리: 1개


In [ ]:
# ======================================================================
# 전체 조합 스위프 — subprocess 격리 + Resume + 재시도
#
# 방어선:
#   1) 서브프로세스 격리 → OOM/segfault 에도 메인 커널 생존
#   2) CSV append + completed set → resume 자동 스킵
#   3) y.pkl → 커널 재시작에도 이 셀만 재실행
#   4) per-run timeout
#   5) 워커가 10샘플마다 mem_json flush
#   6) 실패 시 MAX_ATTEMPTS 회까지 재시도
#        - 중간 실패 attempt 의 mem/log 파일은 삭제
#        - 최종 attempt (성공이든 실패든) 의 파일만 보존
#        - attempt_history 컬럼에 각 시도 결과 기록
#   7) 가벼운 조합부터 실행 (seasonal=False → True, stepwise=True → False)
# ======================================================================
import itertools
import subprocess
import pickle
import csv
import json
import time
import sys
import os
import textwrap
from pathlib import Path
from datetime import datetime

# ---- 설정 ------------------------------------------------------------
# 가벼운 값 먼저 → 무거운 값 나중에 (itertools.product 는 첫 인자가 가장 느리게 순환)
SEASONAL_OPTS = [(False, 0), (True, 24)]   # 비계절 먼저 → 계절 s=24 나중
STEPWISE_OPTS = [True, False]              # stepwise 먼저 → grid search 나중
CRITERION_OPTS = ["aic", "bic", "hqic"]
TREND_OPTS = ["n", "c", "t", "ct"]

PER_RUN_TIMEOUT_S = 3600
MEM_INTERVAL_S = 0.05
MAX_ATTEMPTS = 1          # 초기 1회 + 재시도 2회
RETRY_BACKOFF_S = 2.0     # 재시도 전 대기
RESUME = True

combos = list(itertools.product(SEASONAL_OPTS, STEPWISE_OPTS, CRITERION_OPTS, TREND_OPTS))
RUNNERS = ["rustima", "pmdarima"]
total_runs = len(combos) * len(RUNNERS)

# ---- 출력 디렉터리 (resume or new) ----------------------------------
base = Path("bench_results")
base.mkdir(exist_ok=True)

FIELDS = [
    "idx", "library", "seasonal", "s", "stepwise", "criterion", "trend",
    "status", "order", "seasonal_order", "ic_value", "loglik_own",
    "time_s", "mem_baseline_mb", "mem_peak_mb", "mem_delta_mb",
    "attempts", "attempt_history",
    "error", "log_file", "mem_file",
]

OUT_DIR = None
if RESUME:
    for d in sorted(base.glob("sweep_*"), reverse=True):
        if (d / "sweep_results.csv").exists():
            OUT_DIR = d
            print(f"🔁 Resume — 기존 디렉터리: {OUT_DIR}")
            break
if OUT_DIR is None:
    STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
    OUT_DIR = base / f"sweep_{STAMP}"
    print(f"🆕 신규 디렉터리: {OUT_DIR}")

LOG_DIR = OUT_DIR / "logs"
MEM_DIR = OUT_DIR / "mem_samples"
TMP_DIR = OUT_DIR / "_tmp"
LOG_DIR.mkdir(parents=True, exist_ok=True)
MEM_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUT_DIR / "sweep_results.csv"
if not csv_path.exists():
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=FIELDS).writeheader()

# ---- 완료 집합 -------------------------------------------------------
completed = set()
if csv_path.exists():
    with open(csv_path, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            if r.get("status"):
                try:
                    completed.add((int(r["idx"]), r["library"]))
                except (TypeError, ValueError):
                    pass
print(f"이미 완료: {len(completed)} / {total_runs}")

# ---- y pickle --------------------------------------------------------
Y_PKL = OUT_DIR / "y.pkl"
if "y" in dir():
    with open(Y_PKL, "wb") as f:
        pickle.dump(y, f)
    print(f"y pickled → {Y_PKL}  (len={len(y) if hasattr(y,'__len__') else '?'})")
elif Y_PKL.exists():
    print(f"⚠️  커널에 y 미정의. 기존 y.pkl 재사용: {Y_PKL}")
else:
    raise RuntimeError(
        f"y 가 커널에도 없고 {Y_PKL} 도 없습니다. 데이터 로딩 셀을 먼저 실행하세요.")

# ---- 워커 스크립트 ---------------------------------------------------
WORKER = OUT_DIR / "_worker.py"
WORKER.write_text(textwrap.dedent(r"""
    import sys, os, time, json, pickle, threading, traceback, gc
    (tag, lib, s, stepwise, criterion, trend, y_pkl, result_json,
     mem_json, log_txt, interval) = sys.argv[1:12]
    s = int(s); stepwise = stepwise == "1"; interval = float(interval)

    import psutil
    proc = psutil.Process(os.getpid())

    stop = threading.Event()
    samples = []
    baseline = proc.memory_info().rss / 1024 ** 2
    peak = [baseline]

    def _flush_mem():
        with open(mem_json, "w", encoding="utf-8") as f:
            json.dump({"tag": tag, "baseline_mb": baseline,
                       "peak_mb": peak[0], "delta_mb": peak[0] - baseline,
                       "interval_s": interval, "samples": samples,
                       "partial": not stop.is_set()}, f)

    def _sampler():
        t0 = time.perf_counter()
        samples.append((0.0, baseline))
        i = 0
        while not stop.is_set():
            m = proc.memory_info().rss / 1024 ** 2
            samples.append((time.perf_counter() - t0, m))
            if m > peak[0]: peak[0] = m
            i += 1
            if i % 10 == 0:
                try: _flush_mem()
                except Exception: pass
            stop.wait(interval)

    th = threading.Thread(target=_sampler, daemon=True)
    th.start()

    with open(y_pkl, "rb") as f:
        y = pickle.load(f)

    import io, contextlib
    buf = io.StringIO()
    status, err = "ok", ""
    order = seasonal_order = ic_val = loglik_own = None
    elapsed = None
    try:
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            t0 = time.perf_counter()
            if lib == "rustima":
                from rustima import auto_arima as rs_auto_arima
                res = rs_auto_arima(y, s=s, trend=trend, stepwise=stepwise,
                                    trace=True, criterion=criterion)
                order = str(tuple(res.order))
                so = getattr(res, "seasonal_order", None)
                seasonal_order = str(tuple(so)) if so is not None else ""
                ic_val = getattr(res.result, criterion, None)
                loglik_own = (getattr(res.result, "loglik", None)
                              or getattr(res.result, "llf", None)
                              or getattr(res.result, "log_likelihood", None))
            else:
                import pmdarima as pm
                res = pm.auto_arima(
                    y, seasonal=(s > 0), m=s if s > 0 else 1, trend=trend,
                    stepwise=stepwise, suppress_warnings=True,
                    information_criterion=criterion, trace=True)
                order = str(tuple(res.order))
                seasonal_order = str(tuple(res.seasonal_order))
                getter = getattr(res, criterion, None)
                ic_val = getter() if callable(getter) else None
                ar = getattr(res, "arima_res_", None)
                if ar is not None:
                    loglik_own = getattr(ar, "llf", None)
                if loglik_own is None:
                    llf_fn = getattr(res, "llf", None)
                    loglik_own = llf_fn() if callable(llf_fn) else getattr(res, "loglik", None)
            elapsed = time.perf_counter() - t0
    except Exception as e:
        status = "fail"
        err = f"{type(e).__name__}: {e}"
        traceback.print_exc(file=buf)

    stop.set(); th.join()
    _flush_mem()
    with open(log_txt, "w", encoding="utf-8") as f:
        f.write(buf.getvalue())
    with open(result_json, "w", encoding="utf-8") as f:
        json.dump({"status": status, "error": err,
                   "order": order or "", "seasonal_order": seasonal_order or "",
                   "ic_value": ic_val if isinstance(ic_val, (int, float)) else None,
                   "loglik_own": loglik_own if isinstance(loglik_own, (int, float)) else None,
                   "time_s": elapsed,
                   "mem_baseline_mb": baseline,
                   "mem_peak_mb": peak[0],
                   "mem_delta_mb": peak[0] - baseline}, f)
"""), encoding="utf-8")


def _run_one_attempt(cmd, result_json, log_path, tmp_log_path):
    """한 번의 subprocess 시도. (row_dict, log_txt_for_caller_append) 반환."""
    out = {"status": "", "error": "", "order": "", "seasonal_order": "",
           "ic_value": "", "loglik_own": "", "time_s": "",
           "mem_baseline_mb": "", "mem_peak_mb": "", "mem_delta_mb": ""}
    try:
        cp = subprocess.run(cmd, timeout=PER_RUN_TIMEOUT_S,
                            capture_output=True, text=True,
                            encoding="utf-8", errors="replace")
        if result_json.exists():
            d = json.loads(result_json.read_text(encoding="utf-8"))
            out["status"] = d.get("status", "ok")
            out["error"] = (d.get("error") or "")[:500]
            out["order"] = d.get("order", "")
            out["seasonal_order"] = d.get("seasonal_order", "")
            for src_k, dst_k, fmt in (
                ("ic_value", "ic_value", "{:.4f}"),
                ("loglik_own", "loglik_own", "{:.4f}"),
                ("time_s", "time_s", "{:.4f}"),
                ("mem_baseline_mb", "mem_baseline_mb", "{:.2f}"),
                ("mem_peak_mb", "mem_peak_mb", "{:.2f}"),
                ("mem_delta_mb", "mem_delta_mb", "{:.2f}"),
            ):
                v = d.get(src_k)
                out[dst_k] = fmt.format(v) if isinstance(v, (int, float)) else ""
            try: result_json.unlink()
            except OSError: pass
        else:
            out["status"] = "crashed"
            tail = (cp.stderr or cp.stdout or "").splitlines()[-5:]
            out["error"] = (" | ".join(tail))[:500]
            with open(tmp_log_path, "a", encoding="utf-8") as f:
                f.write(f"\n--- SUBPROCESS CRASH (exit={cp.returncode}) ---\n")
                f.write("STDOUT:\n" + (cp.stdout or "") + "\n")
                f.write("STDERR:\n" + (cp.stderr or "") + "\n")
    except subprocess.TimeoutExpired:
        out["status"] = "timeout"
        out["error"] = f"exceeded {PER_RUN_TIMEOUT_S}s"
        with open(tmp_log_path, "a", encoding="utf-8") as f:
            f.write(f"\n--- TIMEOUT after {PER_RUN_TIMEOUT_S}s ---\n")
    except Exception as e:
        out["status"] = "launch_fail"
        out["error"] = f"{type(e).__name__}: {e}"
    return out


# ---- 메인 루프 -------------------------------------------------------
sweep_t0 = time.perf_counter()
run_i = 0
for idx, ((seasonal, s), stepwise, criterion, trend) in enumerate(combos):
    for lib_name in RUNNERS:
        run_i += 1
        if (idx, lib_name) in completed:
            continue
        tag = f"{idx:02d}_{lib_name}_s{s}_sw{int(stepwise)}_{criterion}_{trend}"
        canonical_mem = MEM_DIR / f"{tag}.json"
        canonical_log = LOG_DIR / f"{tag}.log"

        print(f"\n{'=' * 80}")
        print(f"[{run_i:03d}/{total_runs}] {lib_name}  seasonal={seasonal}(s={s}) "
              f"stepwise={stepwise} crit={criterion} trend={trend}")
        print("=" * 80)

        # 시도별 임시 파일
        attempt_mem_paths = []
        attempt_log_paths = []
        attempt_histories = []
        final_out = None

        for attempt in range(1, MAX_ATTEMPTS + 1):
            tmp_mem = TMP_DIR / f"{tag}_try{attempt}.mem.json"
            tmp_log = TMP_DIR / f"{tag}_try{attempt}.log"
            tmp_result = TMP_DIR / f"{tag}_try{attempt}.result.json"

            cmd = [sys.executable, str(WORKER), tag, lib_name, str(s),
                   "1" if stepwise else "0", criterion, trend,
                   str(Y_PKL), str(tmp_result), str(tmp_mem), str(tmp_log),
                   str(MEM_INTERVAL_S)]

            print(f"  ▶ attempt {attempt}/{MAX_ATTEMPTS}")
            out = _run_one_attempt(cmd, tmp_result, canonical_log, tmp_log)
            attempt_mem_paths.append(tmp_mem)
            attempt_log_paths.append(tmp_log)
            attempt_histories.append(out["status"])
            final_out = out

            if out["status"] == "ok":
                print(f"    → ok")
                break
            print(f"    → {out['status']}: {(out['error'] or '')[:140]}")
            if attempt < MAX_ATTEMPTS:
                time.sleep(RETRY_BACKOFF_S)

        # 최종 시도의 mem/log → canonical 경로, 나머지는 삭제
        final_mem = attempt_mem_paths[-1]
        final_log = attempt_log_paths[-1]
        try:
            if canonical_mem.exists(): canonical_mem.unlink()
            if final_mem.exists(): final_mem.replace(canonical_mem)
        except OSError: pass
        try:
            if canonical_log.exists(): canonical_log.unlink()
            if final_log.exists(): final_log.replace(canonical_log)
        except OSError: pass
        for mp, lp in zip(attempt_mem_paths[:-1], attempt_log_paths[:-1]):
            for p in (mp, lp):
                try:
                    if p.exists(): p.unlink()
                except OSError: pass

        # CSV row
        row = {k: "" for k in FIELDS}
        row.update({
            "idx": idx, "library": lib_name,
            "seasonal": seasonal, "s": s, "stepwise": stepwise,
            "criterion": criterion, "trend": trend,
            "log_file": str(canonical_log.relative_to(OUT_DIR)),
            "mem_file": str(canonical_mem.relative_to(OUT_DIR)),
            "attempts": len(attempt_histories),
            "attempt_history": ";".join(attempt_histories),
        })
        row.update(final_out)

        with open(csv_path, "a", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=FIELDS).writerow(row)

        print(f"  ▣ final [{row['status']}] attempts={len(attempt_histories)} "
              f"history={row['attempt_history']}")
        if row["status"] == "ok":
            print(f"    order={row['order']} ic={row['ic_value']} "
                  f"loglik={row['loglik_own']} time={row['time_s']}s "
                  f"peak={row['mem_peak_mb']}MB")
        elif row["error"]:
            print(f"    err: {row['error'][:200]}")

        completed.add((idx, lib_name))

elapsed = time.perf_counter() - sweep_t0
print(f"\n✅ 세션 종료 — 누적 완료 {len(completed)}/{total_runs}, "
      f"이번 세션 {elapsed/60:.1f}분 소요")
print(f"   CSV : {csv_path}")
print(f"   로그 : {LOG_DIR}")
print(f"   메모리 : {MEM_DIR}")

🆕 신규 디렉터리: bench_results\sweep_20260426_085642
이미 완료: 0 / 96
y pickled → bench_results\sweep_20260426_085642\y.pkl  (len=43800)

[001/96] rustima  seasonal=False(s=0) stepwise=True crit=aic trend=n
  ▶ attempt 1/1
    → ok
  ▣ final [ok] attempts=1 history=ok
    order=(3, 0, 2) ic=341153.3971 loglik=-170570.6985 time=4.4343s peak=180.38MB

[002/96] pmdarima  seasonal=False(s=0) stepwise=True crit=aic trend=n
  ▶ attempt 1/1


## 2.2. sarimax_rs 엔진으로 재적합(sweep_refit)
- sweep_results.csv 에서 각 (idx, library) 의 order 와 seasonal_order 를 읽고
- sarimax_rs 로 재적합한 후 sweep_refit.csv 에 append
- 이미 저장된 (idx, library_source) 는 자동 스킵 (resume)

In [6]:
# ======================================================================
# sweep-refit — sarimax_rs 엔진으로 재적합해 공정 비교 (aic/bic/hqic/loglik)
#
# 방어선:
#   1) 서브프로세스 격리 → refit OOM/segfault 에도 메인 커널 생존
#   2) per-refit timeout → hang 방지
#   3) 재시도 MAX_REFIT_ATTEMPTS 회, 중간 attempt 로그 삭제, 최종만 보존
#   4) CSV append + resume
#
# 메모리 타임라인은 기록하지 않음 — refit 은 criterion 비교용이지
# 속도/메모리 비교가 아니기 때문.
# ======================================================================
import csv
import pickle
import ast
import time
import json
import subprocess
import sys
import textwrap
from pathlib import Path

MAX_REFIT_ATTEMPTS = 3
PER_REFIT_TIMEOUT_S = 1800      # refit 은 auto_arima 보다 훨씬 빨라야 정상
RETRY_BACKOFF_S = 1.0

csvs = sorted(Path("bench_results").glob("sweep_*/sweep_results.csv"))
assert csvs, "sweep-run 먼저 실행하세요."
OUT_DIR = csvs[-1].parent
main_csv = OUT_DIR / "sweep_results.csv"
refit_csv = OUT_DIR / "sweep_refit.csv"
y_pkl = OUT_DIR / "y.pkl"
assert y_pkl.exists(), f"{y_pkl} 없음. sweep-run 재실행 필요."

LOG_DIR = OUT_DIR / "logs_refit"
TMP_DIR = OUT_DIR / "_tmp_refit"
LOG_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

REFIT_FIELDS = [
    "idx", "library_source", "order", "seasonal_order", "trend", "s",
    "criterion", "status",
    "aic_refit", "bic_refit", "hqic_refit", "loglik_refit",
    "time_s", "attempts", "attempt_history",
    "error", "log_file",
]

if not refit_csv.exists():
    with open(refit_csv, "w", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=REFIT_FIELDS).writeheader()

done = set()
with open(refit_csv, newline="", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        try:
            done.add((int(r["idx"]), r["library_source"]))
        except (ValueError, TypeError):
            pass
print(f"이미 재적합된 건: {len(done)}")


# ---- 워커 스크립트 생성 ---------------------------------------------
WORKER_REFIT = OUT_DIR / "_worker_refit.py"
WORKER_REFIT.write_text(textwrap.dedent(r"""
    import sys, time, json, pickle, traceback, ast

    (y_pkl, result_json, log_txt,
     order_s, seasonal_order_s, trend) = sys.argv[1:7]

    def _parse(s):
        if not s or s == "None":
            return None
        try:
            v = ast.literal_eval(s)
            return v if isinstance(v, tuple) else None
        except Exception:
            return None

    order = _parse(order_s)
    seasonal_order = _parse(seasonal_order_s)

    with open(y_pkl, "rb") as f:
        y_data = pickle.load(f)

    import io, contextlib
    buf = io.StringIO()
    status, err = "ok", ""
    aic = bic = hqic = loglik = None
    elapsed = None
    try:
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            from rustima import SARIMAXModel
            kw = dict(order=order, trend=trend)
            if seasonal_order is not None and len(seasonal_order) == 4:
                kw["seasonal_order"] = seasonal_order
            t0 = time.perf_counter()
            m = SARIMAXModel(y_data, **kw)
            r = m.fit()
            elapsed = time.perf_counter() - t0
            aic = getattr(r, "aic", None)
            bic = getattr(r, "bic", None)
            hqic = getattr(r, "hqic", None)
            loglik = (getattr(r, "loglik", None)
                      or getattr(r, "llf", None)
                      or getattr(r, "log_likelihood", None))
    except Exception as e:
        status = "fail"
        err = f"{type(e).__name__}: {e}"
        traceback.print_exc(file=buf)

    with open(log_txt, "w", encoding="utf-8") as f:
        f.write(buf.getvalue())
    with open(result_json, "w", encoding="utf-8") as f:
        json.dump({
            "status": status, "error": err,
            "aic_refit":    aic    if isinstance(aic,    (int, float)) else None,
            "bic_refit":    bic    if isinstance(bic,    (int, float)) else None,
            "hqic_refit":   hqic   if isinstance(hqic,   (int, float)) else None,
            "loglik_refit": loglik if isinstance(loglik, (int, float)) else None,
            "time_s": elapsed,
        }, f)
"""), encoding="utf-8")


def _run_refit_attempt(order, seasonal_order, trend, tmp_result, tmp_log):
    out = {"status": "", "error": "",
           "aic_refit": "", "bic_refit": "", "hqic_refit": "", "loglik_refit": "",
           "time_s": ""}
    cmd = [sys.executable, str(WORKER_REFIT), str(y_pkl),
           str(tmp_result), str(tmp_log),
           str(order), str(seasonal_order), trend]
    try:
        cp = subprocess.run(cmd, timeout=PER_REFIT_TIMEOUT_S,
                            capture_output=True, text=True,
                            encoding="utf-8", errors="replace")
        if tmp_result.exists():
            d = json.loads(tmp_result.read_text(encoding="utf-8"))
            out["status"] = d.get("status", "ok")
            out["error"] = (d.get("error") or "")[:500]
            for src_k, fmt in (
                ("aic_refit", "{:.4f}"), ("bic_refit", "{:.4f}"),
                ("hqic_refit", "{:.4f}"), ("loglik_refit", "{:.4f}"),
                ("time_s", "{:.4f}"),
            ):
                v = d.get(src_k)
                out[src_k] = fmt.format(v) if isinstance(v, (int, float)) else ""
            try: tmp_result.unlink()
            except OSError: pass
        else:
            out["status"] = "crashed"
            tail = (cp.stderr or cp.stdout or "").splitlines()[-5:]
            out["error"] = (" | ".join(tail))[:500]
            with open(tmp_log, "a", encoding="utf-8") as f:
                f.write(f"\n--- SUBPROCESS CRASH (exit={cp.returncode}) ---\n")
                f.write("STDOUT:\n" + (cp.stdout or "") + "\n")
                f.write("STDERR:\n" + (cp.stderr or "") + "\n")
    except subprocess.TimeoutExpired:
        out["status"] = "timeout"
        out["error"] = f"exceeded {PER_REFIT_TIMEOUT_S}s"
        with open(tmp_log, "a", encoding="utf-8") as f:
            f.write(f"\n--- TIMEOUT after {PER_REFIT_TIMEOUT_S}s ---\n")
    except Exception as e:
        out["status"] = "launch_fail"
        out["error"] = f"{type(e).__name__}: {e}"
    return out


def _parse_tup(s):
    if not s or s == "None":
        return None
    try:
        v = ast.literal_eval(s)
        return v if isinstance(v, tuple) else None
    except Exception:
        return None


# ---- 소스 테이블 로드 ------------------------------------------------
rows_by_idx = {}
with open(main_csv, newline="", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        if r.get("status") != "ok":
            continue
        try:
            idx = int(r["idx"])
        except (ValueError, TypeError):
            continue
        rows_by_idx.setdefault(idx, {})[r["library"]] = r

total = sum(len(v) for v in rows_by_idx.values())
print(f"재적합 대상: {total} 건 (sweep_results.csv 중 status=ok 만)")
print(f"로그 : {LOG_DIR}")

t_start = time.perf_counter()
for idx in sorted(rows_by_idx.keys()):
    for lib, src_row in rows_by_idx[idx].items():
        if (idx, lib) in done:
            continue
        order = _parse_tup(src_row.get("order"))
        seasonal_order = _parse_tup(src_row.get("seasonal_order"))
        trend = src_row.get("trend", "n")
        if order is None:
            continue

        tag = f"{idx:02d}_{lib}_refit"
        canonical_log = LOG_DIR / f"{tag}.log"

        print(f"\n[refit] idx={idx:02d} lib={lib:8s} order={order} "
              f"seasonal={seasonal_order} trend={trend}")

        attempt_log_paths = []
        histories = []
        final_out = None

        for attempt in range(1, MAX_REFIT_ATTEMPTS + 1):
            tmp_log = TMP_DIR / f"{tag}_try{attempt}.log"
            tmp_result = TMP_DIR / f"{tag}_try{attempt}.result.json"

            print(f"  ▶ attempt {attempt}/{MAX_REFIT_ATTEMPTS}")
            out = _run_refit_attempt(order, seasonal_order, trend,
                                     tmp_result, tmp_log)
            attempt_log_paths.append(tmp_log)
            histories.append(out["status"])
            final_out = out

            if out["status"] == "ok":
                print(f"    → ok  aic={out['aic_refit']} loglik={out['loglik_refit']} "
                      f"time={out['time_s']}s")
                break
            print(f"    → {out['status']}: {(out['error'] or '')[:140]}")
            if attempt < MAX_REFIT_ATTEMPTS:
                time.sleep(RETRY_BACKOFF_S)

        # 최종 attempt 의 로그만 canonical 경로로, 나머지는 삭제
        final_log = attempt_log_paths[-1]
        try:
            if canonical_log.exists(): canonical_log.unlink()
            if final_log.exists(): final_log.replace(canonical_log)
        except OSError: pass
        for lp in attempt_log_paths[:-1]:
            try:
                if lp.exists(): lp.unlink()
            except OSError: pass

        row = {k: "" for k in REFIT_FIELDS}
        row.update({
            "idx": idx, "library_source": lib,
            "order": str(order),
            "seasonal_order": str(seasonal_order) if seasonal_order else "",
            "trend": trend, "s": src_row.get("s", ""),
            "criterion": src_row.get("criterion", ""),
            "attempts": len(histories),
            "attempt_history": ";".join(histories),
            "log_file": str(canonical_log.relative_to(OUT_DIR)),
        })
        row.update(final_out)

        with open(refit_csv, "a", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=REFIT_FIELDS).writerow(row)
        print(f"  ▣ final [{row['status']}] history={row['attempt_history']}")
        done.add((idx, lib))

elapsed = time.perf_counter() - t_start
print(f"\n✅ 재적합 완료 — 누적 {len(done)} 건, 이번 세션 {elapsed/60:.1f}분")
print(f"   CSV : {refit_csv}")


이미 재적합된 건: 0
재적합 대상: 65 건 (sweep_results.csv 중 status=ok 만)
로그 : bench_results\sweep_20260424_202726\logs_refit

[refit] idx=00 lib=rustima  order=(3, 0, 2) seasonal=(0, 0, 0, 0) trend=n
  ▶ attempt 1/3
    → ok  aic=341153.3971 loglik=-170570.6985 time=0.4301s
  ▣ final [ok] history=ok

[refit] idx=00 lib=pmdarima order=(3, 1, 2) seasonal=(0, 0, 0, 0) trend=n
  ▶ attempt 1/3
    → ok  aic=338935.2211 loglik=-169461.6106 time=0.2252s
  ▣ final [ok] history=ok

[refit] idx=01 lib=rustima  order=(1, 0, 0) seasonal=(0, 0, 0, 0) trend=c
  ▶ attempt 1/3
    → ok  aic=343447.9065 loglik=-171720.9532 time=0.1310s
  ▣ final [ok] history=ok

[refit] idx=01 lib=pmdarima order=(5, 1, 4) seasonal=(0, 0, 0, 0) trend=c
  ▶ attempt 1/3
    → ok  aic=338927.9083 loglik=-169452.9541 time=0.9003s
  ▣ final [ok] history=ok

[refit] idx=02 lib=rustima  order=(2, 0, 2) seasonal=(0, 0, 0, 0) trend=t
  ▶ attempt 1/3
    → ok  aic=343874.6398 loglik=-171931.3199 time=0.4875s
  ▣ final [ok] history=ok

[refit]

## 2.3. 요약(sweep summary)
- Table 1: criterion 값 + 속도 (각 패키지 자체 보고값) + 각각의 차이
- Table 2: 선택된 order + 각 패키지 자체 loglikelihood 값 + 그 차이
- Table 3: 동일 엔진(sarimax_rs) 재적합 criterion 공정 비교

In [15]:
# 3 테이블 요약 + 실패 조합 키워드
#   Table 1 — criterion 값 + 속도 + 피크 메모리 (각 패키지 자체 보고값)
#   Table 2 — 선택된 order + 각 패키지 자체 loglik
#   Table 3 — 동일 엔진(sarimax_rs) 재적합 criterion 공정 비교
import pandas as pd
import numpy as np
from pathlib import Path

csvs = sorted(Path("bench_results").glob("sweep_*/sweep_results.csv"))
assert csvs, "sweep-run 먼저 실행하세요."
latest = csvs[-1]
out_dir = latest.parent
print(f"Source : {latest}")


def _reason(status, error):
    if status == "ok":
        return ""
    if status == "timeout":
        return "TIMEOUT"
    if status == "crashed":
        return "CRASH"
    if status == "launch_fail":
        return "LAUNCH"
    e = (error or "")
    el = e.lower()
    if "memoryerror" in el or ("memory" in el and ("allocat" in el or "alloc" in el)):
        return "OOM"
    if "overflow" in el:
        return "OVERFLOW"
    if "did not converge" in el or "convergence" in el or "converge" in el:
        return "NOCONV"
    if "singular" in el:
        return "SINGULAR"
    if "linalg" in el or "linear algebra" in el:
        return "LINALG"
    if "nan" in el or "inf" in el:
        return "NAN"
    if "timeout" in el:
        return "TIMEOUT"
    return "FAIL"


# ---- 로드 ------------------------------------------------------------
df = pd.read_csv(latest)
for c in ("time_s", "mem_baseline_mb", "mem_peak_mb", "mem_delta_mb",
          "ic_value", "loglik_own"):
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
df["error"] = df.get("error", "").fillna("") if "error" in df.columns else ""
df["reason"] = df.apply(lambda r: _reason(r.get("status", ""), r.get("error", "")), axis=1)

key_cols = ["idx", "seasonal", "s", "stepwise", "criterion", "trend"]


def _lib_frame(lib, prefix):
    value_cols = [c for c in ("order", "seasonal_order", "ic_value", "time_s",
                               "mem_peak_mb", "loglik_own", "status", "error",
                               "reason", "attempts", "attempt_history")
                  if c in df.columns]
    sub = df[df.library == lib][key_cols + value_cols].copy()
    rename = {c: f"{prefix}_{c}" for c in value_cols}
    return sub.rename(columns=rename)


merged = _lib_frame("rustima", "rs").merge(
    _lib_frame("pmdarima", "pm"), on=key_cols, how="outer"
).sort_values("idx").reset_index(drop=True)


# ---- 포맷 헬퍼 --------------------------------------------------------
def _fmt_cell(val, reason, fmt):
    has_val = pd.notna(val)
    if reason:
        if has_val:
            try:
                return f"{fmt.format(val)} [{reason}]"
            except Exception:
                return f"[{reason}]"
        return f"[{reason}]"
    if not has_val:
        return ""
    try:
        return fmt.format(val)
    except Exception:
        return str(val)


def _format_table(tbl, value_fmts):
    out = tbl.copy()
    for col, (fmt, reason_col) in value_fmts.items():
        if col not in out.columns:
            continue
        if reason_col and reason_col in out.columns:
            out[col] = [_fmt_cell(v, r, fmt) for v, r in zip(tbl[col], tbl[reason_col])]
        else:
            out[col] = out[col].apply(lambda v: fmt.format(v) if pd.notna(v) else "")
    return out


# ---- Table 1 — criterion + 속도 + 피크 메모리 ------------------------
t1 = merged[key_cols + ["rs_ic_value", "pm_ic_value", "rs_time_s", "pm_time_s",
                         "rs_mem_peak_mb", "pm_mem_peak_mb",
                         "rs_reason", "pm_reason"]].copy()
t1["delta_ic"]  = t1["pm_ic_value"]    - t1["rs_ic_value"]
t1["speedup"]   = t1["pm_time_s"]      / t1["rs_time_s"]
t1["mem_ratio"] = t1["pm_mem_peak_mb"] / t1["rs_mem_peak_mb"]
t1 = t1.rename(columns={"rs_mem_peak_mb": "rs_mem_peak",
                          "pm_mem_peak_mb": "pm_mem_peak"})

t1_display = _format_table(t1, {
    "rs_ic_value": ("{:.2f}",  "rs_reason"),
    "pm_ic_value": ("{:.2f}",  "pm_reason"),
    "delta_ic":    ("{:+.2f}", None),
    "rs_time_s":   ("{:.3f}",  "rs_reason"),
    "pm_time_s":   ("{:.3f}",  "pm_reason"),
    "speedup":     ("{:.2f}x", None),
    "rs_mem_peak": ("{:.1f}",  "rs_reason"),
    "pm_mem_peak": ("{:.1f}",  "pm_reason"),
    "mem_ratio":   ("{:.2f}x", None),
})
t1_display = t1_display[key_cols + [
    "rs_ic_value", "pm_ic_value", "delta_ic",
    "rs_time_s", "pm_time_s", "speedup",
    "rs_mem_peak", "pm_mem_peak", "mem_ratio",
]]

# 단위 어노테이션
t1_display = t1_display.rename(columns={
    "rs_ic_value": "rs_crit_value",
    "pm_ic_value": "pm_crit_value",
    "delta_ic":    "delta_crit",
    "rs_time_s":   "rs_time (s)",
    "pm_time_s":   "pm_time (s)",
    "speedup":     "speedup (×)",
    "rs_mem_peak": "rs_mem_peak (MB)",
    "pm_mem_peak": "pm_mem_peak (MB)",
    "mem_ratio":   "mem_ratio (×)",
})

# ---- Table 2 — order + 자체 loglik ----------------------------------
t2 = merged[key_cols + ["rs_order", "pm_order", "rs_loglik_own", "pm_loglik_own",
                         "rs_reason", "pm_reason"]].copy()
t2["same_order"]   = (t2["rs_order"].fillna("") == t2["pm_order"].fillna("")) & \
                     (t2["rs_order"].notna() & t2["pm_order"].notna())
t2["delta_loglik"] = t2["rs_loglik_own"] - t2["pm_loglik_own"]
t2 = t2.rename(columns={"rs_loglik_own": "rs_loglik",
                          "pm_loglik_own": "pm_loglik"})

t2_display = _format_table(t2, {
    "rs_loglik":    ("{:.2f}",  "rs_reason"),
    "pm_loglik":    ("{:.2f}",  "pm_reason"),
    "delta_loglik": ("{:+.2f}", None),
})
t2_display["rs_order"] = [f"[{r}]" if r else (str(o) if pd.notna(o) else "")
                          for o, r in zip(t2["rs_order"], t2["rs_reason"])]
t2_display["pm_order"] = [f"[{r}]" if r else (str(o) if pd.notna(o) else "")
                          for o, r in zip(t2["pm_order"], t2["pm_reason"])]
t2_display = t2_display[key_cols + [
    "rs_order", "pm_order", "same_order",
    "rs_loglik", "pm_loglik", "delta_loglik",
]]

# ---- Table 3 — sarimax_rs 재적합 criterion --------------------------
t3 = None
t3_display = None
refit_path = out_dir / "sweep_refit.csv"
if refit_path.exists():
    rf = pd.read_csv(refit_path)
    for c in ("aic_refit", "bic_refit", "hqic_refit", "loglik_refit"):
        if c in rf.columns:
            rf[c] = pd.to_numeric(rf[c], errors="coerce")
    rf["error"] = rf.get("error", "").fillna("") if "error" in rf.columns else ""
    rf["reason"] = rf.apply(
        lambda r: _reason(r.get("status", ""), r.get("error", "")), axis=1)

    def _refit_lib(lib, prefix):
        cols = [c for c in ("aic_refit", "bic_refit", "hqic_refit",
                             "loglik_refit", "reason")
                if c in rf.columns]
        sub = rf[rf.library_source == lib][["idx"] + cols].copy()
        rename = {c: f"{prefix}_{c}" for c in cols if c != "reason"}
        if "reason" in cols:
            rename["reason"] = f"{prefix}_reason_refit"
        return sub.rename(columns=rename)

    t3 = (merged[key_cols + ["rs_order", "pm_order", "rs_reason", "pm_reason"]]
          .merge(_refit_lib("rustima", "rs"), on="idx", how="left")
          .merge(_refit_lib("pmdarima", "pm"), on="idx", how="left"))

    def _pick_crit(row, prefix):
        col = f"{prefix}_{row['criterion']}_refit"
        return row[col] if col in row.index else np.nan

    t3["rs_crit_refit"]    = t3.apply(lambda r: _pick_crit(r, "rs"), axis=1)
    t3["pm_crit_refit"]    = t3.apply(lambda r: _pick_crit(r, "pm"), axis=1)
    t3["delta_crit_refit"] = t3["pm_crit_refit"] - t3["rs_crit_refit"]

    t3["rs_final_reason"] = t3.apply(
        lambda r: r["rs_reason"] if r["rs_reason"]
        else (r.get("rs_reason_refit", "") or ""), axis=1)
    t3["pm_final_reason"] = t3.apply(
        lambda r: r["pm_reason"] if r["pm_reason"]
        else (r.get("pm_reason_refit", "") or ""), axis=1)

    t3_display = _format_table(t3, {
        "rs_crit_refit":    ("{:.2f}",  "rs_final_reason"),
        "pm_crit_refit":    ("{:.2f}",  "pm_final_reason"),
        "delta_crit_refit": ("{:+.2f}", None),
    })
    t3_display["rs_order"] = [f"[{r}]" if r else (str(o) if pd.notna(o) else "")
                              for o, r in zip(t3["rs_order"], t3["rs_final_reason"])]
    t3_display["pm_order"] = [f"[{r}]" if r else (str(o) if pd.notna(o) else "")
                              for o, r in zip(t3["pm_order"], t3["pm_final_reason"])]
    t3_display = t3_display[key_cols + [
        "rs_order", "pm_order",
        "rs_crit_refit", "pm_crit_refit", "delta_crit_refit",
    ]]

# ---- 저장 ------------------------------------------------------------
p1 = out_dir / "table1_criterion_speed.csv"
p2 = out_dir / "table2_order_loglik.csv"
p3 = out_dir / "table3_refit_criterion.csv"
t1_display.to_csv(p1, index=False)
t2_display.to_csv(p2, index=False)
if t3_display is not None:
    t3_display.to_csv(p3, index=False)

# ---- 실패 요약 -------------------------------------------------------
fail_df = df[df["reason"] != ""][
    ["idx", "library", "seasonal", "stepwise", "criterion", "trend",
     "status", "reason", "attempts", "attempt_history", "error"]
].copy() if "attempts" in df.columns else df[df["reason"] != ""].copy()
fail_path = out_dir / "failures.csv"
fail_df.to_csv(fail_path, index=False)

# ---- 출력 ------------------------------------------------------------
with pd.option_context("display.max_rows", 200, "display.width", 260):
    print("\n" + "=" * 100)
    print("Table 1 — criterion + 속도 + 피크 메모리   (Δ = pm-rs,  speedup/mem_ratio = pm/rs)")
    print("=" * 100)
    print(t1_display.to_string(index=False))

    print("\n" + "=" * 100)
    print("Table 2 — 선택된 order + 각 패키지 자체 loglik   (Δloglik = rs - pm)")
    print("=" * 100)
    print(t2_display.to_string(index=False))

    if t3_display is not None:
        print("\n" + "=" * 100)
        print("Table 3 — 동일 엔진(sarimax_rs) 재적합 criterion   (Δ = pm - rs)")
        print("=" * 100)
        print(t3_display.to_string(index=False))
    else:
        print("\n⚠️  Table 3 생략: sweep-refit 셀을 먼저 실행하세요.")

    if len(fail_df):
        print("\n" + "=" * 100)
        print(f"실패/스킵 목록 ({len(fail_df)}건) — keyword 별 구분")
        print("=" * 100)
        print(fail_df.to_string(index=False))
    else:
        print("\n✨ 실패 런 없음.")

print(f"\n저장 파일:")
print(f"  T1  {p1}")
print(f"  T2  {p2}")
if t3_display is not None:
    print(f"  T3  {p3}")
print(f"  F   {fail_path}")

Source : bench_results\sweep_20260424_202726\sweep_results.csv

Table 1 — criterion + 속도 + 피크 메모리   (Δ = pm-rs,  speedup/mem_ratio = pm/rs)
 idx  seasonal  s  stepwise criterion trend rs_crit_value pm_crit_value delta_crit rs_time (s) pm_time (s) speedup (×) rs_mem_peak (MB) pm_mem_peak (MB) mem_ratio (×)
   0     False  0      True       aic     n     341153.40     338935.22   -2218.18       4.427     145.869      32.95x            162.9           1729.5        10.62x
   1     False  0      True       aic     c     343447.91     338928.99   -4518.92       2.397     757.696     316.09x            167.3           3717.6        22.22x
   2     False  0      True       aic     t     343874.64     343411.51    -463.13       5.977     141.259      23.63x            167.2           2023.0        12.10x
   3     False  0      True       aic    ct     343705.94     343413.51    -292.43      10.388     163.841      15.77x            174.7           2033.8        11.64x
   4     False  0      Tr

## 2.4. 메모리 타임라인(sweep visulization)
- 시각화: 대시보드 PNG + 히트맵 + 메모리 타임라인 + HTML 리포트

In [24]:
import json
import base64
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# 한글 폰트 (Windows: Malgun Gothic)
plt.rcParams["font.family"] = ["Malgun Gothic", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

csvs = sorted(Path("bench_results").glob("sweep_*/sweep_results.csv"))
assert csvs, "먼저 sweep-run 셀을 실행해 결과를 생성하세요."
latest = csvs[-1]
out_dir = latest.parent
plot_dir = out_dir / "plots"
tl_dir = plot_dir / "timelines"
plot_dir.mkdir(exist_ok=True)
tl_dir.mkdir(exist_ok=True)
print(f"Source : {latest}")
print(f"Plots  : {plot_dir}")

df = pd.read_csv(latest)
for c in ["time_s", "mem_baseline_mb", "mem_peak_mb", "mem_delta_mb"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
ok = df[df.status == "ok"].copy()
def _make_combo(r):
    return f"s{r.s}_sw{int(r.stepwise)}_{r.criterion}_{r.trend}"

df["combo"] = df.apply(_make_combo, axis=1)
ok["combo"] = ok.apply(_make_combo, axis=1)

# 비교 그래프는 s=0 (두 패키지 모두 성공한 조합)만 사용
ok_cmp = ok[ok.s == 0].copy()
rs_cmp = ok_cmp[ok_cmp.library == "rustima"].set_index("combo")
pm_cmp = ok_cmp[ok_cmp.library == "pmdarima"].set_index("combo")
common = sorted(rs_cmp.index.intersection(pm_cmp.index))

# 메모리 타임라인은 OOM/실패 런도 OOM 직전까지의 RSS 샘플이 mem_file 에 남으므로
# status 와 무관하게 mem_file 이 있는 조합을 모두 포함
common_all = sorted(df.loc[df["mem_file"].notna(), "combo"].unique().tolist())

LIB_COLORS = {"rustima": "#5bc0de", "pmdarima": "#d9534f"}

# ------------------------------------------------------------------
# 막대그래프 — 조합별 실행시간 (s=0)
# ------------------------------------------------------------------
bars_time_s0_path = None
if common:
    pivot_t = ok_cmp.pivot_table(index="combo", columns="library", values="time_s")
    pivot_t = pivot_t.reindex(common)
    cols_present = [c for c in ["rustima", "pmdarima"] if c in pivot_t.columns]
    pivot_t = pivot_t[cols_present]
    colors = [LIB_COLORS[c] for c in cols_present]

    fig, ax = plt.subplots(figsize=(max(10, 0.45 * len(common) + 6), 5.5))
    pivot_t.plot(kind="bar", ax=ax, color=colors)
    ax.legend(framealpha=0.4)
    ax.set_ylabel("time (s)")
    ax.set_title("조합별 실행 시간")
    ax.set_xlabel("")
    ax.tick_params(axis="x", labelrotation=80, labelsize=7)
    ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    bars_time_s0_path = plot_dir / "bars_time_s0.png"
    fig.savefig(bars_time_s0_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print("  saved bars_time_s0.png")


# ------------------------------------------------------------------
# 막대그래프 — 조합별 피크 메모리 (s=0)
# ------------------------------------------------------------------
bars_mem_s0_path = None
if common:
    pivot_m = ok_cmp.pivot_table(index="combo", columns="library", values="mem_peak_mb")
    pivot_m = pivot_m.reindex(common)
    cols_present = [c for c in ["rustima", "pmdarima"] if c in pivot_m.columns]
    pivot_m = pivot_m[cols_present]
    colors = [LIB_COLORS[c] for c in cols_present]

    fig, ax = plt.subplots(figsize=(max(10, 0.45 * len(common) + 6), 5.5))
    pivot_m.plot(kind="bar", ax=ax, color=colors)
    ax.legend(framealpha=0.4)
    ax.set_ylabel("peak RSS (MB)")
    ax.set_title("조합별 피크 메모리 — s=0")
    ax.set_xlabel("")
    ax.tick_params(axis="x", labelrotation=80, labelsize=7)
    ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    bars_mem_s0_path = plot_dir / "bars_mem_s0.png"
    fig.savefig(bars_mem_s0_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print("  saved bars_mem_s0.png")


# ------------------------------------------------------------------
# 막대그래프 — 조합별 피크 메모리 (s=24)
#   pmdarima 가 OOM 으로 실패해도 mem_peak_mb 는 OOM 직전까지 측정된 값이라
#   비교 가능. status 무관하게 df 사용.
# ------------------------------------------------------------------
bars_s24_path = None
df_s24 = df[df.s == 24].copy()
if len(df_s24):
    combos_s24 = sorted(df_s24["combo"].unique())
    pivot_m24 = df_s24.pivot_table(index="combo", columns="library",
                                    values="mem_peak_mb", aggfunc="first")
    pivot_m24 = pivot_m24.reindex(combos_s24)
    cols_present = [c for c in ["rustima", "pmdarima"] if c in pivot_m24.columns]
    pivot_m24 = pivot_m24[cols_present]
    colors = [LIB_COLORS[c] for c in cols_present]

    fig, ax = plt.subplots(figsize=(max(10, 0.45 * len(combos_s24) + 6), 5.5))
    pivot_m24.plot(kind="bar", ax=ax, color=colors)
    ax.legend(framealpha=0.4)
    ax.set_ylabel("peak RSS (MB)")
    ax.set_title("조합별 피크 메모리 — s=24")
    ax.set_xlabel("")
    ax.tick_params(axis="x", labelrotation=80, labelsize=7)
    ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    bars_s24_path = plot_dir / "bars_per_combo_s24.png"
    fig.savefig(bars_s24_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print("  saved bars_per_combo_s24.png")


# ------------------------------------------------------------------
# 그룹별(stepwise / criterion / trend) 비교 — 박스플롯
#   막대그래프는 분포 정보를 잃고 조합 수가 많아질수록 가독성↓
#   각 그룹 내 분포(중앙값·IQR·이상치)를 한눈에 보여주는 박스플롯이 적합
# ------------------------------------------------------------------
def _grouped_boxplot(group_col, fname_stem, title_label, legend_loc="upper left"):
    cats = sorted(ok_cmp[group_col].dropna().unique().tolist(), key=lambda x: str(x))
    if not cats:
        return None
    fig, axes = plt.subplots(1, 2, figsize=(max(8, 1.8 * len(cats) + 4), 5))
    width = 0.36
    x = np.arange(len(cats))

    for ax_idx, (metric, ylabel, ax_title) in enumerate([
        ("mem_peak_mb", "peak RSS (MB)", f"피크 메모리 — {title_label}별"),
        ("time_s",      "time (s)",      f"실행 시간 — {title_label}별"),
    ]):
        ax = axes[ax_idx]
        for i, lib in enumerate(["rustima", "pmdarima"]):
            data = [ok_cmp[(ok_cmp.library == lib) & (ok_cmp[group_col] == c)][metric]
                    .dropna().values for c in cats]
            offset = (i - 0.5) * width
            bp = ax.boxplot(
                data, positions=x + offset, widths=width * 0.9,
                patch_artist=True, showfliers=True,
                medianprops=dict(color="black", linewidth=1.4),
                flierprops=dict(marker="o", markersize=3, alpha=0.5),
            )
            for box in bp["boxes"]:
                box.set(facecolor=LIB_COLORS[lib], alpha=0.6, edgecolor="black")
            ax.plot([], [], color=LIB_COLORS[lib], lw=8, alpha=0.6, label=lib)

        ax.set_xticks(x)
        ax.set_xticklabels([str(c) for c in cats])
        ax.set_xlabel(title_label)
        ax.set_ylabel(ylabel)
        ax.set_title(ax_title)
        ax.legend(loc=legend_loc, framealpha=0.4)
        ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    out = plot_dir / f"compare_{fname_stem}.png"
    fig.savefig(out, dpi=120, bbox_inches="tight")
    plt.close(fig)
    return out


grouped_plots = []
for col, stem, label, legend_loc in [
    ("stepwise",  "by_stepwise",  "stepwise",  "upper left"),
    ("criterion", "by_criterion", "criterion", "upper left"),
    ("trend",     "by_trend",     "trend",     "upper right"),
]:
    p = _grouped_boxplot(col, stem, label, legend_loc=legend_loc)
    if p:
        grouped_plots.append((p, label))
print("  saved grouped comparison boxplots")


# ------------------------------------------------------------------
# 메모리 타임라인 — 한 장(그리드) + 조합별 개별 파일 (전 구간)
# ------------------------------------------------------------------
def _load_samples(mem_file):
    p = out_dir / mem_file
    if not p.exists():
        return [], []
    d = json.loads(p.read_text(encoding="utf-8"))
    s = d.get("samples") or []
    if not s:
        return [], []
    t, m = zip(*s)
    return list(t), list(m)


n = len(common_all)
if n:
    ncols = 6
    nrows = int(np.ceil(n / ncols))
    fig_g, axes_g = plt.subplots(nrows, ncols,
                                 figsize=(ncols * 3.2, nrows * 2.2),
                                 sharex=False)
    axes_flat = np.atleast_1d(axes_g).flatten()
    for ax in axes_flat[n:]:
        ax.axis("off")
    for i, combo in enumerate(common_all):
        ax = axes_flat[i]
        for lib, color in [("rustima", "#5bc0de"), ("pmdarima", "#d9534f")]:
            row = df[(df.combo == combo) & (df.library == lib)]
            if row.empty:
                continue
            t, m = _load_samples(row.iloc[0]["mem_file"])
            if t:
                ax.plot(t, m, color=color, label=lib, lw=1.1)
        ax.set_title(combo, fontsize=8)
        ax.tick_params(labelsize=6)
        ax.grid(alpha=0.25)
        if i == 0:
            ax.legend(fontsize=7, framealpha=0.4)
    fig_g.suptitle("메모리 타임라인 — 전체 조합", fontsize=13, y=1.002)
    fig_g.supxlabel("elapsed (s)", fontsize=9)
    fig_g.supylabel("RSS (MB)", fontsize=9)
    fig_g.tight_layout()
    fig_g.savefig(plot_dir / "timelines_grid.png", dpi=110, bbox_inches="tight")
    plt.close(fig_g)
    print("  saved timelines_grid.png")

    for combo in common_all:
        fig, ax = plt.subplots(figsize=(9, 3.6))
        for lib, color in [("rustima", "#5bc0de"), ("pmdarima", "#d9534f")]:
            row = df[(df.combo == combo) & (df.library == lib)]
            if row.empty:
                continue
            t, m = _load_samples(row.iloc[0]["mem_file"])
            if t:
                peak = row.iloc[0]["mem_peak_mb"]
                dur = row.iloc[0]["time_s"]
                status = row.iloc[0]["status"]
                tag = "" if status == "ok" else f" [{status.upper()}]"
                ax.plot(t, m, color=color,
                        label=f"{lib}{tag}  peak={peak:.1f}MB  time={dur:.2f}s",
                        lw=1.3)
        ax.set_xlabel("elapsed (s)")
        ax.set_ylabel("RSS (MB)")
        ax.set_title(f"메모리 타임라인 — {combo}")
        # combo prefix: "s0_..." → upper left,  그 외(예: "s24_...") → upper right
        ax.legend(loc="upper left" if combo.startswith("s0_") else "upper right", framealpha=0.4)
        ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(tl_dir / f"timeline_{combo}.png", dpi=100,
                    bbox_inches="tight")
        plt.close(fig)
    print(f"  saved {len(common_all)} per-combo timelines")


# ------------------------------------------------------------------
# HTML 리포트 — Table 1/2/3 → 실패 런 → 막대 → 박스 → 타임라인 순
# ------------------------------------------------------------------
def _img_b64(p):
    return base64.b64encode(p.read_bytes()).decode("ascii")


def _read_table_html(p, caption):
    if not p.exists():
        return ""
    tbl = pd.read_csv(p)
    return f"<h2>{caption}</h2>" + tbl.to_html(index=False, na_rep="")


def _table1_summary_html():
    """Table 1 의 6 개 수치 컬럼에 대한 max/min 요약 (raw df 기반)."""
    pivot = df.pivot_table(
        index=["idx", "seasonal", "s", "stepwise", "criterion", "trend"],
        columns="library",
        values=["time_s", "mem_peak_mb"],
        aggfunc="first",
    )
    if pivot.empty:
        return ""

    def col(metric, lib):
        try:
            return pivot[(metric, lib)]
        except KeyError:
            return pd.Series(dtype=float)

    rs_time   = col("time_s",      "rustima")
    pm_time   = col("time_s",      "pmdarima")
    rs_mem    = col("mem_peak_mb", "rustima")
    pm_mem    = col("mem_peak_mb", "pmdarima")
    speedup   = pm_time / rs_time
    mem_ratio = pm_mem  / rs_mem

    # rs_time 은 s=0 조합 내에서만 min/max 판단 (s=24 는 rs 자체가 오래 걸려 비교 의미↓)
    rs_time_s0 = rs_time[rs_time.index.get_level_values("s") == 0]

    metrics = [
        ("rs_time (s)",      rs_time_s0, "{:.3f}"),
        ("pm_time (s)",      pm_time,    "{:.3f}"),
        ("speedup (×)",      speedup,    "{:.2f}x"),
        ("rs_mem_peak (MB)", rs_mem,     "{:.1f}"),
        ("pm_mem_peak (MB)", pm_mem,     "{:.1f}"),
        ("mem_ratio (×)",    mem_ratio,  "{:.2f}x"),
    ]

    def _fmt(v, fmt):
        return fmt.format(v) if pd.notna(v) else ""

    summary = pd.DataFrame(
        {label: [_fmt(s.min(), fmt), _fmt(s.max(), fmt)]
         for label, s, fmt in metrics},
        index=["min", "max"],
    )
    return "<h3>Table 1 — max / min 요약</h3>" + summary.to_html()


REPORT_TITLE = "Rustima vs Statsmodels Report - 2021.04~2026.03 SMP Data"

html = [
    "<!doctype html><html><head><meta charset='utf-8'>",
    f"<title>{REPORT_TITLE}</title><style>",
    "body{font-family:system-ui,'Malgun Gothic',sans-serif;max-width:1400px;margin:20px auto;padding:0 16px;color:#222}",
    "h1,h2,h3{border-bottom:1px solid #ddd;padding-bottom:4px}",
    "img{max-width:100%;border:1px solid #ddd;border-radius:4px;margin:8px 0}",
    "table{border-collapse:collapse;font-size:13px;margin:10px 0}",
    "th,td{border:1px solid #ccc;padding:4px 8px;text-align:right}",
    "th{background:#f4f4f4}",
    "details{margin:8px 0}",
    "summary{cursor:pointer;font-weight:600}",
    "</style></head><body>",
    f"<h1>{REPORT_TITLE}</h1>",
]

# ---- Table 1/2/3 (보고서 맨 앞) ------------------------------------
html.append(_read_table_html(out_dir / "table1_criterion_speed.csv",
                             "Table 1 — criterion + 속도 + 피크 메모리"))
html.append(_table1_summary_html())
html.append(_read_table_html(out_dir / "table2_order_loglik.csv",
                             "Table 2 — 선택된 order + 자체 loglik"))
html.append(_read_table_html(out_dir / "table3_refit_criterion.csv",
                             "Table 3 — 동일 엔진(sarimax_rs) 재적합 criterion"))

# ---- 실패 런 (Table 3 직후) ----------------------------------------
n_fail = int((df.status == "fail").sum())
if n_fail:
    html.append("<h2>실패 런</h2>")
    html.append(df[df.status == "fail"][
        ["idx", "library", "seasonal", "stepwise", "criterion", "trend", "error"]
    ].to_html(index=False))


def _embed(p, title):
    if not p or not p.exists():
        return
    html.append(f"<h2>{title}</h2>")
    html.append(f"<img src='data:image/png;base64,{_img_b64(p)}'>")


# ---- 막대그래프 (박스플롯 직전) ------------------------------------
_embed(bars_mem_s0_path,  "조합별 피크 메모리 — s=0")
_embed(bars_s24_path,     "조합별 피크 메모리 — s=24")
_embed(bars_time_s0_path, "조합별 실행시간")

for p, label in grouped_plots:
    _embed(p, f"{label}별 비교 — 실행시간 / 피크 메모리")

_embed(plot_dir / "timelines_grid.png", "메모리 타임라인 — 전체 조합")

if common_all:
    html.append("<h2>조합별 메모리 타임라인</h2>")
    for combo in common_all:
        p = tl_dir / f"timeline_{combo}.png"
        if p.exists():
            html.append(f"<details><summary>{combo}</summary>"
                        f"<img src='data:image/png;base64,{_img_b64(p)}'></details>")

html.append("</body></html>")
html_path = out_dir / "summary.html"
html_path.write_text("\n".join(html), encoding="utf-8")
print(f"\n✅ HTML 리포트: {html_path.resolve()}")
print(f"   브라우저에서 열면 모든 차트/테이블이 한 장에 보입니다.")

Source : bench_results\sweep_20260424_202726\sweep_results.csv
Plots  : bench_results\sweep_20260424_202726\plots
  saved bars_time_s0.png
  saved bars_mem_s0.png
  saved bars_per_combo_s24.png
  saved grouped comparison boxplots
  saved timelines_grid.png
  saved 41 per-combo timelines

✅ HTML 리포트: C:\claude\Rust-python-arima\rustima\bench_results\sweep_20260424_202726\summary.html
   브라우저에서 열면 모든 차트/테이블이 한 장에 보입니다.


## 10. 결론

| 항목 | 설명 |
|------|------|
| **우도 유사도** | `\|ΔLL\|` 값을 확인 — 대부분 < 1.0이면 양 엔진이 동일 최적점 수렴 |
| **수렴 속도** | `Speedup` 열 — sarimax_rs가 statsmodels 대비 몇 배 빠른지 |
| **파라미터 일치** | `Max\|Δparam\|` — 0.01 미만이면 사실상 동일 |
| **auto_arima 차수** | 동일 차수 선택 시 order_match=True, 다를 경우 refit AIC로 비교 |